# Rug Pull Detection on Solana — Data Cleaning, EDA, Chronological Splitting, Missing-Data Preprocessing, Feature Importance, and Artifact Export

## Scope and Experimental Definition

This notebook prepares and audits three early-observation datasets for Solana rug-pull
prediction: q1 = 750 s, q2 = 2127 s, and q3 = 5585 s after token creation. The outcome
horizon is 24 hours.

A binary rug-pull label is used only when `label_status == VALID`. A positive label requires
all three rule components: a liquidity drop greater than 80%, a price drop greater than 90%,
and persistent liquidity depletion for at least 80% of the 30-minute confirmation interval,
with liquidity remaining below the persistence threshold at the endpoint.

`IN_WINDOW_EVENT` and `UNKNOWN_*` outcomes are retained for audit but excluded from binary
supervised modelling. Therefore, class 0 means that the token does not satisfy the
operational rug-pull definition within the 24-hour outcome horizon; it should not be
interpreted as proof that the token is globally benign.

Both label construction and graph extraction are token-mint-level analyses. `lp_address`
is retained as an identifying field, but the historical liquidity series and recovered
transfer activity are collected for the token mint rather than for one specific liquidity
pool. Results should therefore be interpreted as token-level early rug-pull detection,
not pool-specific prediction.

The notebook performs data cleaning, cross-table consistency checks, exploratory analysis,
structural leakage checks, graph diagnostics, chronological splitting, leakage-safe
missing-data preprocessing, training-only model-based feature importance, and final
model-ready artifact export.

All complete-data descriptive EDA is performed before the model-development split is
used. After those analyses and leakage audits are complete, one central chronological
train/validation/test assignment is created at the token-mint level and reused across
q1/q2/q3 and across tabular and graph-based model representations.

The split is label-independent for allocation and targets 70% training, 15% validation,
and 15% test data in chronological order. Because the outcome label uses future
information, samples immediately preceding validation or test are embargoed when their
labels would not yet be available at the beginning of the next partition.

Learned missing-value preprocessing and model-based feature importance are then fitted
using training data only. Validation and test values do not influence fitted imputation
parameters or the feature-importance ranking.

Graph analyses are conditional on recovered transfer activity. Because transfer discovery
is based on the mint-address signature history, the recovered graphs are partial
representations of observable transfer activity even when an edge-collection job completes
successfully.

# Installing and Loading Libraries

In [1]:
%pip install -q pandas numpy matplotlib seaborn networkx pyarrow scikit-learn joblib

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os
import sys
import json
import joblib
import sklearn
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt      # NOTE: .pyplot — `import matplotlib as plt` has no .subplots()
import seaborn as sns
import networkx as nx

from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.ensemble import HistGradientBoostingClassifier

sns.set_theme(style="whitegrid")
pd.set_option('display.max_columns', 50)
pd.set_option('display.width', 150)

# Loading Dataset

In [3]:
PATH = './data/'

NODES_FILE = 'graph_nodes.csv'
EDGES_FILE = 'graph_edges.csv'
LABELS_FILE = 'rule_based_labels.csv'

EXPERIMENTS = ['q1', 'q2', 'q3']

In [4]:
nodes = pd.read_csv(PATH + NODES_FILE)
edges = pd.read_csv(PATH + EDGES_FILE)
labels = pd.read_csv(PATH + LABELS_FILE)

print('nodes ', nodes.shape)
print('edges ', edges.shape)
print('labels', labels.shape)

FileNotFoundError: [Errno 2] No such file or directory: './data/graph_nodes.csv'

In [ ]:
print('\nNODE DTYPES')
display(nodes.dtypes)

print('\nEDGE DTYPES')
display(edges.dtypes)

print('\nLABEL DTYPES')
display(labels.dtypes)

In [ ]:
nodes.head()

In [ ]:
edges.head()

In [ ]:
labels.head()

# Cleaning Data

In [ ]:
nodes.describe()

In [ ]:
edges.describe()

In [ ]:
labels.describe()

## Step 1 — Clean each table

Each input table is cleaned separately. Row counts before and after cleaning are printed
to make filtering explicit and auditable.

In [ ]:
def clean_nodes(df):
    """Drop blanks/duplicates, keep only labelled rows, parse the extra columns."""
    n0 = len(df)
    df = df.dropna(how='all')
    # collector may re-run an experiment and append; keep the freshest row
    df = df.drop_duplicates(subset=['lp_address', 'token_mint', 'experiment_id'], keep='last')

    # graph_nodes should already contain only VALID rows, but make sure
    df = df[df.label_status == 'VALID']
    df = df[df.rug_pull_label.notna()]
    df['rug_pull_label'] = df.rug_pull_label.astype(int)

    # authority fields are recovered from the selected mint-creation transaction.
    # convert their presence into initial-authority indicators
    df['has_initial_mint_authority'] = df.mint_authority.notna()
    df['has_initial_freeze_authority'] = df.freeze_authority.notna()

    # how long AFTER the window closes does the label become known? must be > 0
    df['label_lead_time'] = df.label_available_time - df.observation_end

    invalid_time = df.label_lead_time.isna() | (df.label_lead_time <= 0)
    leaking = invalid_time.sum()
    if leaking:
        print(f'  WARNING: {leaking} rows with invalid label timing (missing or <= 0 lead time) - dropped')
        print(df[invalid_time][['token_mint', 'observation_end', 'label_available_time']].head())
        df = df[~invalid_time]

    print(f'clean_nodes : {n0} -> {len(df)} rows')
    return df.reset_index(drop=True)

In [ ]:
def clean_edges(df):
    """Drop blanks, duplicate transfers, self-loops and out-of-window transfers."""
    n0 = len(df)
    df = df.dropna(how='all')
    df = df.dropna(subset=['lp_address', 'experiment_id', 'token_mint', 'tx_signature', 'source_address', 
                           'target_address', 'timestamp', 'amount', 'observation_start', 'observation_end'])

    # IMPORTANT: one tx_signature can contain MANY transfers, so it is not a key by itself
    df = df.drop_duplicates(subset=['lp_address', 'experiment_id', 'token_mint', 'tx_signature', 
                                    'source_address', 'target_address', 'amount'])


    # self-loops are removed because they add no connectivity between distinct recovered endpoints
    df = df[df.source_address != df.target_address]                    # self-loops
    df = df[(df.timestamp >= df.observation_start) &
            (df.timestamp < df.observation_end)]                       # leakage guard

    df['t_frac'] = ((df.timestamp - df.observation_start) /
                    (df.observation_end - df.observation_start))       # relative position inside the observation window: 0 <= t_frac < 1

    print(f'clean_edges : {n0} -> {len(df)} rows')
    return df.reset_index(drop=True)

In [ ]:
def clean_labels(df):
    """Only dedupe. This table is the audit trail, so unlabelled rows are KEPT."""
    n0 = len(df)
    df = df.dropna(how='all')
    df = df.drop_duplicates(subset=['lp_address', 'token_mint', 'experiment_id'], keep='last')

    print(f'clean_labels: {n0} -> {len(df)} rows')
    return df.reset_index(drop=True)

In [ ]:
nodes = clean_nodes(nodes)
edges = clean_edges(edges)
labels = clean_labels(labels)

In [ ]:
print('\nCLEANED DATASET SIZES')
print(f'nodes : {nodes.shape}')
print(f'labels: {labels.shape}')
print(f'edges : {edges.shape}')

print('\nLABEL STATUS COUNTS')
display(labels['label_status'].value_counts(dropna=False))

print('\nNODE LABEL COUNTS')
display(nodes['rug_pull_label'].value_counts(dropna=False).sort_index())

In [ ]:
KEY = ['lp_address', 'token_mint', 'experiment_id']

valid_labels = labels[
    (labels['label_status'] == 'VALID') &
    labels['rug_pull_label'].notna()
].copy()

label_node_audit = valid_labels[
    KEY + [
        'rug_pull_label',
        'token_creation_time',
        'observation_window_seconds',
        'observation_end_time',
        'label_available_time'
    ]
].merge(
    nodes[
        KEY + [
            'rug_pull_label',
            'creation_time',
            'observation_window_seconds',
            'observation_end',
            'label_available_time'
        ]
    ],
    on=KEY,
    how='outer',
    indicator=True,
    suffixes=('_label', '_node')
)

print(label_node_audit['_merge'].value_counts())

display(
    label_node_audit.loc[
        label_node_audit['_merge'] != 'both',
        KEY + ['_merge']
    ]
)

In [ ]:
matched = label_node_audit[label_node_audit['_merge'] == 'both']

print('label mismatch:', (matched['rug_pull_label_label'] != matched['rug_pull_label_node']).sum())
print('creation_time mismatch:', (matched['token_creation_time'] != matched['creation_time']).sum())
print('window length mismatch:', (matched['observation_window_seconds_label'] != matched['observation_window_seconds_node']).sum())
print('observation_end mismatch:', (matched['observation_end_time'] != matched['observation_end']).sum())
print('label_available_time mismatch:', (matched['label_available_time_label'] != matched['label_available_time_node']).sum())

assert (label_node_audit['_merge'] == 'right_only').sum() == 0
assert (matched['rug_pull_label_label'] != matched['rug_pull_label_node']).sum() == 0
assert (matched['token_creation_time'] != matched['creation_time']).sum() == 0
assert (matched['observation_window_seconds_label'] != matched['observation_window_seconds_node']).sum() == 0
assert (matched['observation_end_time'] != matched['observation_end']).sum() == 0
assert (matched['label_available_time_label'] != matched['label_available_time_node']).sum() == 0

**<span style="color:red">Observation:</span>** One token has VALID labels for q1/q2/q3
but no corresponding rows in `graph_nodes.csv`. Under the node-collector logic, ordinary
missing numerical features do not by themselves suppress a node row; incomplete features
are retained as missing values.

Therefore, these three missing rows are recorded as a persisted-node-output discrepancy
rather than ordinary feature missingness. The exact cause cannot be determined from the
current artifacts and would require the collector logs or progress state. All 5,332
matched rows agree on label and temporal metadata.

## Step 2 — Separate by Observation Window

q1/q2/q3 represent three different observation-window lengths. Many tokens appear in
more than one observation window, so separate tables are created for each experiment
before the remaining analyses are performed.

This separation is not a train/validation/test split. The central chronological
model-development split is defined later in this notebook, after the complete-data
descriptive analyses and leakage audits.

In [ ]:
def separate_qs(df, expr_id):
    """Take one experiment's slice. experiment_id is dropped: it is now a property of the table."""
    out = df[df.experiment_id == expr_id].drop(columns='experiment_id')
    return out.reset_index(drop=True)

In [ ]:
nodes_q = {q: separate_qs(nodes, q) for q in EXPERIMENTS}
edges_q = {q: separate_qs(edges, q) for q in EXPERIMENTS}
labels_q = {q: separate_qs(labels, q) for q in EXPERIMENTS}

for q in EXPERIMENTS:
    print(f'{q}: {len(nodes_q[q]):5} tokens, {len(edges_q[q]):7} transfers, '
          f'window = {nodes_q[q].observation_window_seconds.iloc[0]} s')

## Step 3 — The `full` and `panel` token scopes

For post-observation supervised prediction, a rug-pull event that occurs inside the
observation window cannot serve as a future binary target. Longer windows therefore
exclude more early events. Additional samples can also be unavailable because of
`UNKNOWN_*` label states. In the current model-ready node tables the token sets are nested
(q3 ⊂ q2 ⊂ q1), so the common-token panel is the q3 token set.

* **full** = every available supervised node sample for this observation window
* **panel** = only supervised node samples shared across q1/q2/q3

Both are reported: `full` preserves all available supervised node samples for each
observation window, while `panel` enables a controlled comparison of window length using
the same tokens.

In [ ]:
q1_tokens = set(nodes_q['q1'].token_mint)
q2_tokens = set(nodes_q['q2'].token_mint)
q3_tokens = set(nodes_q['q3'].token_mint)

print('q2 subset of q1:', q2_tokens.issubset(q1_tokens))
print('q3 subset of q2:', q3_tokens.issubset(q2_tokens))

assert q2_tokens.issubset(q1_tokens)
assert q3_tokens.issubset(q2_tokens)

In [ ]:
common_tokens = set.intersection(*[set(nodes_q[q].token_mint) for q in EXPERIMENTS])
print(f'common tokens: {len(common_tokens)}')

# what does the panel throw away?
for q in EXPERIMENTS:
    df = nodes_q[q]
    dropped = df[~df.token_mint.isin(common_tokens)]
    print(f'{q}: drops {len(dropped):4} tokens | rug rate of dropped = '
          f'{dropped.rug_pull_label.mean() if len(dropped) else float("nan"):.2f}')

**<span style="color:red">Observation:</span>** The common-token panel is strongly selected toward tokens that survive
the longer observation windows. The samples removed from q1 and q2 have much higher rug
rates, so `full` is retained as the primary analysis scope and `panel` is used only for
controlled cross-window comparisons.

In [ ]:
panel_meta = pd.concat([
    nodes_q[q]
    .loc[
        nodes_q[q]['token_mint'].isin(common_tokens),
        ['token_mint', 'lp_address', 'rug_pull_label', 'creation_time']
    ]
    .assign(experiment_id=q)
    for q in EXPERIMENTS
])

print('panel tokens with inconsistent labels:', (panel_meta.groupby('token_mint')['rug_pull_label'].nunique() > 1).sum())
print('panel tokens with inconsistent LP address:', (panel_meta.groupby('token_mint')['lp_address'].nunique() > 1).sum())
print('panel tokens with inconsistent creation_time:', (panel_meta.groupby('token_mint')['creation_time'].nunique() > 1).sum())

assert (panel_meta.groupby('token_mint')['rug_pull_label'].nunique() <= 1).all()
assert (panel_meta.groupby('token_mint')['lp_address'].nunique() <= 1).all()
assert (panel_meta.groupby('token_mint')['creation_time'].nunique() <= 1).all()

In [ ]:
def apply_scope(df, scope):
    """Apply the predefined full or common-token panel scope."""
    if scope == 'full':
        return df

    if scope == 'panel':
        return df[df.token_mint.isin(common_tokens)].reset_index(drop=True)

    raise ValueError("scope must be either 'full' or 'panel'")

## Step 4 — Sanity checks

These checks verify key uniqueness, observation-window integrity, label timing, edge
membership, and basic numerical validity. Any failed check raises an `AssertionError`
and stops downstream processing.

In [ ]:
def run_checks(nodes_df, edges_df, labels_df, name):
    checks = {}

    # ------------------------------------------------------------
    # Model-ready node population
    # ------------------------------------------------------------
    node_keys = pd.MultiIndex.from_frame(
        nodes_df[
            ['lp_address', 'token_mint']
        ].drop_duplicates()
    )

    # ------------------------------------------------------------
    # VALID labelled population
    # Edges are collected directly from these rows, not from
    # graph_nodes.csv.
    # ------------------------------------------------------------
    valid_labels = labels_df[
        (labels_df['label_status'] == 'VALID')
        &
        (labels_df['rug_pull_label'].isin([0, 1]))
    ].copy()

    label_keys = pd.MultiIndex.from_frame(
        valid_labels[
            ['lp_address', 'token_mint']
        ].drop_duplicates()
    )

    label_windows = (
        valid_labels[
            [
                'lp_address',
                'token_mint',
                'token_creation_time',
                'observation_end_time'
            ]
        ]
        .rename(
            columns={
                'token_creation_time': 'observation_start',
                'observation_end_time': 'observation_end'
            }
        )
        .drop_duplicates()
    )

    label_window_keys = pd.MultiIndex.from_frame(
        label_windows[
            [
                'lp_address',
                'token_mint',
                'observation_start',
                'observation_end'
            ]
        ]
    )

    # ------------------------------------------------------------
    # Recovered-edge population
    # ------------------------------------------------------------
    edge_keys = pd.MultiIndex.from_frame(
        edges_df[
            ['lp_address', 'token_mint']
        ].drop_duplicates()
    )

    edge_window_keys = pd.MultiIndex.from_frame(
        edges_df[
            [
                'lp_address',
                'token_mint',
                'observation_start',
                'observation_end'
            ]
        ].drop_duplicates()
    )

    edge_dedup_cols = [
        'lp_address',
        'token_mint',
        'tx_signature',
        'source_address',
        'target_address',
        'amount'
    ]

    # ------------------------------------------------------------
    # Node-level checks
    # ------------------------------------------------------------
    checks['token key is unique'] = (
        not nodes_df.duplicated(
            ['lp_address', 'token_mint']
        ).any()
    )

    checks['token_mint is unique'] = (not nodes_df.duplicated('token_mint').any())

    checks['one window length'] = nodes_df.observation_window_seconds.nunique() == 1

    checks['window timestamps match duration'] = ((nodes_df.observation_end - nodes_df.observation_start) == nodes_df.observation_window_seconds).all()

    checks['observation starts at creation time'] = (nodes_df.observation_start == nodes_df.creation_time).all()

    checks['labels available after cutoff'] = (nodes_df.label_lead_time > 0).all()

    checks['labels are binary'] = nodes_df.rug_pull_label.isin([0, 1]).all()

    checks['node rows belong to VALID labels'] = node_keys.isin(label_keys).all()

    # ------------------------------------------------------------
    # Edge-level checks
    # ------------------------------------------------------------
    checks['edges inside window'] = ((edges_df.t_frac >= 0) & (edges_df.t_frac < 1)).all()

    checks['edges belong to VALID labelled samples'] = edge_keys.isin(label_keys).all()

    checks['edge windows match VALID label windows'] = edge_window_keys.isin(label_window_keys).all()

    checks['no self-loop edges'] = (edges_df.source_address != edges_df.target_address).all()

    checks['edge dedup key is unique'] = ~edges_df.duplicated(edge_dedup_cols).any()

    checks['edge window duration positive'] = (edges_df.observation_end > edges_df.observation_start).all()

    checks['edge amounts finite and non-negative'] = np.isfinite(edges_df['amount']).all() & (edges_df['amount'] >= 0).all()

    # ------------------------------------------------------------
    # Audit: valid recovered-edge samples that do not have a
    # persisted graph_nodes.csv row.
    #
    # This is reported, not treated as an edge-data failure.
    # ------------------------------------------------------------
    edge_sample_keys = (
        edges_df[
            ['lp_address', 'token_mint']
        ]
        .drop_duplicates()
    )

    node_sample_keys = (
        nodes_df[
            ['lp_address', 'token_mint']
        ]
        .drop_duplicates()
    )

    edge_without_node = (
        edge_sample_keys
        .merge(
            node_sample_keys,
            on=['lp_address', 'token_mint'],
            how='left',
            indicator=True
        )
        .query('_merge == "left_only"')
        .drop(columns='_merge')
        .reset_index(drop=True)
    )

    print(f'\n[{name}]')

    for k, v in checks.items():
        print(f'  {"OK  " if v else "FAIL"} {k}')

    print(f'  INFO recovered-edge samples without a persisted node row: {len(edge_without_node)}')

    if len(edge_without_node):
        display(edge_without_node)

    if not all(checks.values()):
        raise AssertionError(f'{name}: one or more sanity checks failed')

    return checks

In [ ]:
for q in EXPERIMENTS:
    run_checks(nodes_q[q], edges_q[q], labels_q[q], q)

### Fine-Grained Temporal Rug-Pull Prevalence — Descriptive Dataset Audit

To examine temporal non-stationarity without imposing a train/validation/test split,
each observation-window population is sorted by token creation time and divided into
ten consecutive approximately equal-count chronological bins.

This analysis is descriptive dataset accounting only. The bins are not model-development
splits and are not used for feature selection, preprocessing, threshold selection, or
model tuning. Because q1/q2/q3 contain different eligible supervised populations, the
curves are interpreted within each observation window rather than as a same-token
comparison.

In [ ]:
N_CHRONO_BINS = 10

temporal_parts = []

for q in EXPERIMENTS:
    d = nodes_q[q][['creation_time', 'rug_pull_label']].sort_values('creation_time').reset_index(drop=True).copy()

    # Consecutive approximately equal-count chronological bins.
    d['chrono_bin'] = np.floor(np.arange(len(d)) * N_CHRONO_BINS / len(d)).astype(int)

    d['chrono_bin'] = d['chrono_bin'].clip(upper=N_CHRONO_BINS - 1)

    s = d.groupby('chrono_bin').agg(median_creation_time=('creation_time', 'median'), rug_rate=('rug_pull_label', 'mean'), n=('rug_pull_label', 'size')).reset_index()

    s['experiment_id'] = q
    s['bin_number'] = s['chrono_bin'] + 1
    s['date'] = pd.to_datetime(s['median_creation_time'], unit='s', utc=True)

    temporal_parts.append(s)

temporal_prevalence = pd.concat(temporal_parts, ignore_index=True)

peak_idx = temporal_prevalence.groupby('experiment_id')['rug_rate'].idxmax()

temporal_peak_summary = (
    temporal_prevalence
    .loc[
        peak_idx,
        ['experiment_id', 'bin_number', 'date', 'rug_rate', 'n']
    ]
    .sort_values('experiment_id')
    .reset_index(drop=True)
)

temporal_peak_summary['rug_rate_percent'] = temporal_peak_summary['rug_rate'] * 100

display(temporal_peak_summary)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4.8))

for q in EXPERIMENTS:
    p = temporal_prevalence[temporal_prevalence['experiment_id'] == q]

    ax.plot(
        p['date'],
        p['rug_rate'] * 100,
        marker='o',
        linewidth=2,
        label=q
    )

ax.set_ylabel('rug-pull rate (%)')
ax.set_xlabel('token creation time')
ax.set_title('Fine-grained temporal rug-pull prevalence\n(10 equal-count chronological bins per observation window)')
ax.legend(title='observation window')

plt.tight_layout()
plt.show()

**<span style="color:red">Observation:</span>** The chronological bins show that rug-pull
prevalence is not constant over the data-collection period. The exact peak bin, date, and
prevalence for each observation window are reported by the table above.

This analysis is descriptive only. No train/validation/test boundaries are defined here,
and the temporal bins are not used as modelling partitions. Because the bins are
constructed separately within the q1/q2/q3 populations, differences between the three
curves should not be interpreted as same-token effects of observation-window length.

# Exploratory Data Analysis (EDA)

**EDA policy:** The analyses in this section characterize the complete cleaned dataset
before model-development partitions are used. Descriptive label distributions,
missingness, feature distributions, univariate associations, correlations, and graph
structure may therefore be examined on the complete data.

No model-based feature importance or learned missing-value imputation is fitted on the
complete cleaned dataset.

After all complete-data EDA and leakage audits are finished, this notebook creates one
central chronological train/validation/test split. Learned missing-data preprocessing and
model-based feature importance are then fitted using training data only. Validation and
test data do not influence those fitted transformations or the feature-importance ranking.

## Step 1 — Labels and class balance

In [ ]:
def label_summary(nodes_df, labels_df):
    """Rug rate, plus how many tokens the labelling rules could not label."""
    n_rug = int(nodes_df.rug_pull_label.sum())
    n_valid_labels = int(((labels_df['label_status'] == 'VALID') & labels_df['rug_pull_label'].notna()).sum())
    return pd.Series({
        'attempted': len(labels_df),
        'valid_labels': n_valid_labels,
        'model_ready': len(nodes_df),
        'non_rug': len(nodes_df) - n_rug,
        'rug': n_rug,
        'rug_rate': nodes_df.rug_pull_label.mean(),
        'label_coverage': n_valid_labels / len(labels_df),
        'model_ready_coverage': len(nodes_df) / len(labels_df),
        'valid_without_node_row': n_valid_labels - len(nodes_df),
        'in_window_event': int((labels_df.label_status == 'IN_WINDOW_EVENT').sum()),
    })

In [ ]:
balance = pd.DataFrame({q: label_summary(nodes_q[q], labels_q[q]) for q in EXPERIMENTS}).T

balance

In [ ]:
status_counts = pd.DataFrame({
    q: labels_q[q]['label_status'].value_counts(dropna=False)
    for q in EXPERIMENTS
}).fillna(0).astype(int)

status_counts

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))

balance[['non_rug', 'rug']].plot(kind='bar', stacked=True, ax=ax[0], color=['#4c72b0', '#c44e52'])
ax[0].set_title('Model-ready class balance')
ax[0].set_ylabel('tokens')
ax[0].tick_params(axis='x', rotation=0)
ax[0].legend(['non-rug', 'rug'])

ax[1].plot(balance.index, balance.rug_rate * 100, marker='o', color='#c44e52')
ax[1].set_title('Model-ready rug rate by observation window')
ax[1].set_ylabel('rug-pull rate (%)')

plt.tight_layout()
plt.show()

**<span style="color:red">Observation:</span>** Label availability decreases as the
observation window becomes longer: valid-label coverage falls from about 72.7% in q1 to
66.6% in q2 and 58.3% in q3, while `IN_WINDOW_EVENT` cases increase from 128 to 321 and
590. The model-ready rug-pull rate also decreases from about 39.0% to 35.9% and 30.1%.
Therefore, q1/q2/q3 should not be interpreted as identical populations observed for
different durations; the eligible supervised population changes with the cutoff.

## Step 2 — Recovered-Edge Presence

`graph_edges.csv` contains the retained recovered edge rows from the final edge export.
A token-experiment sample with no edge row therefore has no retained recovered transfer
in this export. This does not prove that the token had no on-chain transfers, because
transfer discovery begins from the mint-address signature history and is therefore only
a partial view of token-transfer activity.

The analysis below therefore measures **recovered-edge presence**, not proof of true
transfer absence or complete transfer coverage. Graph diagnostics use only samples with
at least one recovered edge.

In [ ]:
def add_edge_counts(nodes_df, edges_df):
    """Attach the number of recovered transfers present in the current edge file."""
    counts = edges_df.groupby('token_mint').size().rename('n_transfers')
    out = nodes_df.merge(counts, on='token_mint', how='left')
    out['n_transfers'] = out.n_transfers.fillna(0).astype(int)
    out['has_recovered_edges'] = out.n_transfers > 0
    return out


for q in EXPERIMENTS:
    nodes_q[q] = add_edge_counts(nodes_q[q], edges_q[q])
    cov = nodes_q[q].has_recovered_edges
    print(f'{q}: {cov.sum():3} / {len(cov):5} tokens have recovered edges ({100 * cov.mean():.1f}%)')

In [ ]:
# subset with at least one recovered edge, used for graph diagnostics
edge_present = {q: nodes_q[q][nodes_q[q].has_recovered_edges].reset_index(drop=True) for q in EXPERIMENTS}

# Descriptive dataset audit: compare class prevalence in the full supervised data
# with the non-empty recovered-graph subset. This is not used for model tuning.
for q in EXPERIMENTS:
    print(f'{q}: rug rate all = {nodes_q[q].rug_pull_label.mean():.3f} | recovered-edge subset = {edge_present[q].rug_pull_label.mean():.3f}')

In [ ]:
for q in EXPERIMENTS:
    print(f'\n{q} recovered-edge presence by class:')
    display(
        nodes_q[q]
        .groupby('rug_pull_label')['has_recovered_edges']
        .agg(['count', 'sum', 'mean'])
        .rename(index={0: 'non-rug', 1: 'rug'})
    )

**<span style="color:red">Observation:</span>** The presence of at least one recovered
edge is consistently higher among rug-pull samples than among non-rug samples across
q1/q2/q3. This shows that the non-empty recovered-graph subset is class-dependent and
should not be treated as representative of the full supervised dataset. Importantly, 
absence of an edge row does not prove that the token had no on-chain transfers, because
mint-address-based transfer discovery provides only a partial view of token-transfer activity.

### Earliest Recovered-Edge Presence Across Observation Windows — Common-Panel Samples

The marginal recovered-edge rates of q1/q2/q3 are based on different supervised
populations. To compare recovered-edge presence across observation-window lengths while
holding token identity fixed, this analysis uses the common q1/q2/q3 token panel.

For each token, the earliest observation window containing at least one row in the
current recovered-edge table is recorded. This represents earliest recovered-edge
presence, not the token's true first transfer time and not proof of collection
completeness.

Monotonicity across q1/q2/q3 is checked only as an internal consistency property of the
current recovered edge output. It is not treated as evidence that the recovered graph is
complete.

In [ ]:
panel_edge_presence = nodes_q['q3'].loc[nodes_q['q3']['token_mint'].isin(common_tokens), ['token_mint', 'rug_pull_label']].copy()

for q in EXPERIMENTS:
    q_presence = (nodes_q[q][['token_mint', 'has_recovered_edges']].rename(columns={'has_recovered_edges': q}))

    panel_edge_presence = panel_edge_presence.merge(q_presence, on='token_mint', how='left', validate='one_to_one')

assert panel_edge_presence[['q1', 'q2', 'q3']].notna().all().all()

non_monotone = (
    (panel_edge_presence['q1'] & ~panel_edge_presence['q2'])
    |
    (panel_edge_presence['q2'] & ~panel_edge_presence['q3'])
)

print('common-panel tokens with non-monotone recovered-edge presence:', int(non_monotone.sum()))

if non_monotone.any():
    raise RuntimeError('Recovered-edge presence is not monotone across q1/q2/q3. Inspect these cases before interpreting earliest recovered-edge presence.')

In [ ]:
panel_edge_presence['first_recovered_edge_window'] = np.select(
    [
        panel_edge_presence['q1'],
        panel_edge_presence['q2'],
        panel_edge_presence['q3']
    ],
    [
        'q1',
        'q2',
        'q3'
    ],
    default='none by q3'
)

category_order = [
    'q1',
    'q2',
    'q3',
    'none by q3'
]

edge_emergence_counts = (
    pd.crosstab(
        panel_edge_presence['rug_pull_label'],
        panel_edge_presence['first_recovered_edge_window']
    )
    .reindex(
        index=[0, 1],
        columns=category_order,
        fill_value=0
    )
)

edge_emergence_pct = edge_emergence_counts.div(edge_emergence_counts.sum(axis=1), axis=0) * 100

edge_emergence_pct = edge_emergence_pct.rename(index={0: 'non-rug', 1: 'rug'})

In [ ]:
fig, ax = plt.subplots(figsize=(8.5, 4.6))

edge_emergence_pct.plot(
    kind='bar',
    stacked=True,
    ax=ax
)

for container in ax.containers:
    bar_labels = [f'{value:.1f}%' if value >= 5 else '' for value in container.datavalues]

    ax.bar_label(
        container,
        labels=bar_labels,
        label_type='center',
        fontsize=8
    )

ax.set_ylabel('percentage of class (%)')
ax.set_xlabel('')
ax.set_title('Earliest recovered-edge presence\ncommon-panel samples')

ax.tick_params(axis='x', rotation=0)
ax.legend(title='first recovered-edge presence', bbox_to_anchor=(1.02, 1), loc='upper left')

plt.tight_layout()
plt.show()

print('Counts:')
display(edge_emergence_counts.rename(index={0: 'non-rug', 1: 'rug'}))

print('Percentages:')
display(edge_emergence_pct.round(1))

**<span style="color:red">Observation:</span>** The table and stacked bars above show when
recovered-edge presence first becomes observable within the common-token population.
The result should be interpreted only as a property of the current recovered-edge data.

A token categorized as `none by q3` has no retained edge row in the current edge table by
the longest observation cutoff. This does not prove that the token had no transfers, and
recovered-edge presence should not be interpreted as evidence of complete transfer
recovery.

## Step 3 — Token-Level Features

### Missingness

In [ ]:
FEATURES = ['volume_sell_change_percent', 'volume_buy_change_percent',
            'unique_wallet_count', 'unique_wallet_change_percent',
            'liquidity_at_creation', 'liquidity_at_observation_end']

# Check overall feature missingness before interpreting feature distributions or associations
print('Missing features %:')
pd.DataFrame({q: nodes_q[q][FEATURES].isna().mean() * 100 for q in EXPERIMENTS}).round(1)

In [ ]:
def feature_auc(df, col, label='rug_pull_label', min_n=20, min_per_class=10):    
    """Rank AUC of a single feature. 0.5 = no rank separation; values farther from 0.5 indicate stronger univariate separation."""
    d = df[[col, label]].dropna()

    if d[label].nunique() < 2:
        return np.nan          # only one class present -> AUC undefined
    if len(d) < min_n:
        return np.nan          # too few samples to mean anything

    if (d[label].value_counts() < min_per_class).any():
        return np.nan

    pos = d[d[label] == 1][col]
    neg = d[d[label] == 0][col]
    ranks = pd.concat([pos, neg]).rank().values
    
    return (ranks[:len(pos)].sum() - len(pos)*(len(pos)+1)/2) / (len(pos)*len(neg))

### Class-Conditional Missingness — Descriptive Full Dataset

Feature availability is also summarized separately by outcome class. This is a descriptive
data-quality analysis of the complete cleaned dataset and is not used to select predictors
for the final models.

In [ ]:
missing_by_class = {}

for q in EXPERIMENTS:
    for label_value, class_name in [(0, 'non-rug'), (1, 'rug')]:
        subset = nodes_q[q][nodes_q[q]['rug_pull_label'] == label_value]

        missing_by_class[f'{q}\n{class_name}'] = subset[FEATURES].isna().mean() * 100

missing_by_class = pd.DataFrame(missing_by_class)

feature_display_names = {
    'volume_sell_change_percent': 'Sell-volume change',
    'volume_buy_change_percent': 'Buy-volume change',
    'unique_wallet_count': 'Unique-wallet count',
    'unique_wallet_change_percent': 'Unique-wallet change',
    'liquidity_at_creation': 'Liquidity at creation',
    'liquidity_at_observation_end': 'Liquidity at observation end'
}

missing_plot = missing_by_class.rename(index=feature_display_names)

fig, ax = plt.subplots(figsize=(10, 4.8))

sns.heatmap(
    missing_plot,
    annot=True,
    fmt='.1f',
    vmin=0,
    vmax=100,
    cmap='YlOrRd',
    cbar_kws={'label': 'Missing observations (%)'},
    ax=ax
)

ax.set_title('Feature missingness by class and observation window')
ax.set_xlabel('observation window / class')
ax.set_ylabel('feature')

plt.tight_layout()
plt.show()

**<span style="color:red">Observation:</span>** Missingness differs across observation
windows and may also differ between the two outcome classes. The heatmap above is used
only to characterize data availability. These differences should not be interpreted as
causal evidence or used directly to select final predictors.

Missing values are preserved throughout the complete-data EDA. No imputation method is
fitted before the chronological split.

After the central split is fixed later in this notebook, exactly two predefined
missing-data representations are constructed separately for q1, q2, and q3:

1. native missing values (`raw_nan`);
2. training-median imputation with explicit missingness indicators
   (`median_indicator`).

The median-imputation parameters are fitted using training predictors only and then
applied unchanged to validation and test data. The outcome label is never supplied to
the imputation model.

### Single-Feature Rank Association — Descriptive Only

The following univariate rank AUC values describe marginal association in the cleaned
dataset. They are not used for final feature selection or model configuration.

In [ ]:
node_auc = pd.DataFrame({q: {c: feature_auc(nodes_q[q], c) for c in FEATURES}
                            for q in EXPERIMENTS
                        })

node_n = pd.DataFrame({q: {c: nodes_q[q][[c, 'rug_pull_label']].dropna().shape[0] for c in FEATURES}
                          for q in EXPERIMENTS
                      })

pd.concat(
    {
        'auc': node_auc.round(3),
        'n': node_n
    },
    axis=1
)

**<span style="color:red">Observation:</span>** Univariate AUC values must be interpreted
together with their non-missing sample counts. A feature can show apparently strong
separation while being observed for only a small and potentially selected subset of
tokens. These values are descriptive only and do not determine the final feature set.

### Robust Distribution Summaries — Descriptive Full Dataset

In [ ]:
for q in EXPERIMENTS:
    print(f'\n{q} feature medians by class:')
    display(
        nodes_q[q]
        .groupby('rug_pull_label')[FEATURES]
        .median()
        .rename(index={0: 'non-rug', 1: 'rug'})
        .round(3)
    )

    print(f'{q} feature IQR:')
    display(
        nodes_q[q]
        .groupby('rug_pull_label')[FEATURES]
        .quantile([0.25, 0.75])
        .rename(index={0: 'non-rug', 1: 'rug'}, level=0)
        .round(3)
    )

**<span style="color:red">Observation:</span>** The class-conditional medians and
interquartile ranges summarize the observed non-missing values in each observation
window. Because several features have substantial and class-dependent missingness,
these summaries should be interpreted as descriptive conditional patterns rather than
population-level or causal effects.

### Feature Correlations — Descriptive Full Dataset

In [ ]:
# Feature-vs-feature correlation is used as a descriptive redundancy diagnostic.
# No feature is removed in this notebook solely because of pairwise correlation.
corr_q = {
    q: nodes_q[q][FEATURES].corr(
        method='spearman',
        min_periods=30
    )
    for q in EXPERIMENTS
}

for q in EXPERIMENTS:
    fig, ax = plt.subplots(figsize=(7, 6))

    sns.heatmap(
        corr_q[q],
        cmap='RdBu_r',
        center=0,
        vmin=-1,
        vmax=1,
        annot=True,
        fmt='.2f',
        square=True,
        ax=ax
    )

    ax.set_title(f'{q} feature correlation (Spearman)')

    plt.tight_layout()
    plt.show()

# flag strongly correlated feature pairs for redundancy review in each observation window
for q in EXPERIMENTS:
    print(f'\n{q}:')
    corr_current = corr_q[q]

    for i in range(len(FEATURES)):
        for j in range(i + 1, len(FEATURES)):
            r = corr_current.iloc[i, j]

            if pd.notna(r) and abs(r) > 0.8:
                print(f'high correlation: {FEATURES[i]} <-> {FEATURES[j]} : {r:+.2f}')

**<span style="color:red">Observation:</span>** The correlation structure changes across
observation windows. q1 shows strong association between liquidity at creation and at the
observation cutoff, whereas q3 shows strong correlation among the volume-change features
and wallet-change percentage.

In q3, several correlations involving `liquidity_at_creation` are left undefined because
the pairwise number of non-missing observations is below the `min_periods=30` threshold
used for Spearman correlation. Correlations are therefore reported only where sufficient
pairwise observations are available.

The reported correlations are redundancy diagnostics only; no feature is removed solely
because of pairwise correlation, particularly given the unequal missingness patterns.

### Initial Authority Metadata

In [ ]:
AUTHORITY_EDA_FEATURES = [
    'has_initial_mint_authority',
    'has_initial_freeze_authority'
]

# Initial-authority indicators are retained for EDA only.
# A missing authority field cannot currently distinguish a true absent authority
# from incomplete recovery of the mint-initialization metadata.
#
# Legacy variable name: these are token-window/tabular predictors,
# not transfer-graph node attributes.
NODE_MODEL_FEATURES = FEATURES.copy()

for q in EXPERIMENTS:
    print(f'\n{q} initial-authority prevalence (%):')
    display(
        nodes_q[q]
        .groupby('rug_pull_label')[AUTHORITY_EDA_FEATURES]
        .mean()
        .mul(100)
        .rename(index={0: 'non-rug', 1: 'rug'})
        .round(1)
    )

**<span style="color:red">Observation:</span>** `has_initial_mint_authority` is nearly
constant in the cleaned data, whereas `has_initial_freeze_authority` shows greater class
separation. These indicators describe initial authorities parsed from the selected
mint-creation transaction, not the token's later query-time authority state. Because an
empty authority field cannot currently distinguish a genuinely absent authority from
incomplete recovery of initialization metadata, both authority indicators are retained
for EDA only and excluded from `NODE_MODEL_FEATURES`.

### Creator Metadata

In [ ]:
for q in EXPERIMENTS:
    creator_values = nodes_q[q]['creators'].fillna('').astype(str).str.strip()

    empty_creator_metadata = creator_values.isin(['', '[]'])

    print(
        q,
        'empty creator metadata % =',
        round(empty_creator_metadata.mean() * 100, 1),
        '| unique non-empty creator values =',
        creator_values[~empty_creator_metadata].nunique()
    )

**<span style="color:red">Observation:</span>** In this dataset, `creators` denotes
non-mint signer accounts recovered from the selected mint-creation transaction; it is not
Metaplex creator metadata. An empty list (`[]`) should therefore be interpreted as empty
recovered creator-side metadata, not necessarily as proof that no creator account existed.
The field is also high-cardinality, so raw creator identity is intentionally excluded from
`NODE_MODEL_FEATURES` to avoid address memorization.

### Feature-Selection and Missing-Data Boundary

No model-based feature importance or learned missing-value imputation is fitted during
the complete-data EDA above.

After the complete-data EDA and leakage audits, this notebook creates one central
chronological split. Missing-data variants are then fitted and generated from that fixed
split, and model-based feature importance is computed using training data only.

The resulting training-only feature-importance rankings are saved as final experimental
artifacts and reported in the EDA report. They are not recomputed later.

The two missing-data representations are also saved as final model-ready artifacts:
`raw_nan` and `median_indicator`. The median imputer is fitted on training predictors
only and applied unchanged to validation and test data.

High-missingness predictors such as `liquidity_at_creation` remain in the candidate
registry and are not removed solely because of missingness.

## Step 4 — Graph features

Within each observation-window dataset, one graph-feature row is constructed for each token with at least one recovered edge.
These summaries describe the recovered transfer graph and provide both diagnostic graph statistics and
candidate classical baseline features.

Graph endpoints are resolved owner addresses when owner metadata is available; otherwise the
underlying token-account address is retained. Therefore, `n_endpoints`, `n_senders`, and
`n_receivers` count recovered graph endpoints and should not be interpreted as pure wallet
counts.

Only features listed in `GRAPH_MODEL_FEATURES` are retained as candidate modelling
predictors. Amount-based features are diagnostic only because the edge collector may fall
back to raw base units when decimal metadata is unavailable.

Because transfer discovery begins from the mint-address signature history, the recovered
graph is a partial representation of observable token transfer activity even when a
collection job completes successfully. Graph statistics should therefore not be interpreted
as complete token-wide transfer summaries.

In [ ]:
def graph_features(edges_df):
    """Per-token summary of the recovered transfer graph."""
    g = edges_df.groupby('token_mint')

    # Sender/receiver counts refer to recovered endpoint addresses. An endpoint may be
    # a resolved owner address or, when owner metadata is unavailable, a token-account address.
    out = g.agg(
        n_transfers=('tx_signature', 'size'),
        n_senders=('source_address', 'nunique'),
        n_receivers=('target_address', 'nunique'),
        amount_sum=('amount', 'sum'),
        amount_median=('amount', 'median'),
        first_t_frac=('t_frac', 'min'),
        mean_t_frac=('t_frac', 'mean'),
    )

    # Endpoints are resolved owner addresses when owner metadata is available.
    # otherwise the underlying token-account address is retained.
    endpoints = pd.concat([
        edges_df[['token_mint', 'source_address']].rename(columns={'source_address': 'endpoint'}),
        edges_df[['token_mint', 'target_address']].rename(columns={'target_address': 'endpoint'}),
    ])
    out['n_endpoints'] = endpoints.groupby('token_mint').endpoint.nunique()

    # distinct directed endpoint pairs -> density
    pairs = edges_df.groupby(['token_mint', 'source_address', 'target_address']).size()
    out['n_edges'] = pairs.groupby('token_mint').size()

    out['density'] = out.n_edges / (out.n_endpoints * (out.n_endpoints - 1))
    out.loc[out.n_endpoints < 2, 'density'] = np.nan

    # average total degree (in-degree + out-degree) in the simple directed graph
    out['avg_degree'] = 2 * out.n_edges / out.n_endpoints

    # Diagnostic concentration measure based on recovered transfer amounts.
    # Some collector rows may use scaled token units while rows lacking decimal
    # metadata fall back to raw base units, so this feature is not used for modelling.
    sent = edges_df.groupby(['token_mint', 'source_address']).amount.sum()
    top_sent = sent.groupby('token_mint').max()
    total_sent = sent.groupby('token_mint').sum().replace(0, np.nan)
    out['top_sender_share'] = top_sent / total_sent

    return out.reset_index()

In [ ]:
gfeat = {}

for q in EXPERIMENTS:
    gf = graph_features(edges_q[q])

    gfeat[q] = gf.merge(
        edge_present[q][
            [
                'lp_address',
                'token_mint',
                'creation_time',
                'rug_pull_label'
            ]
        ],
        on='token_mint',
        how='inner',
        validate='one_to_one'
    )

    print(f'{q}: {gfeat[q].shape[0]} tokens x {gfeat[q].shape[1]} columns')

gfeat['q1'].head()

In [ ]:
# Graph-feature AUCs are descriptive univariate associations computed on
# all cleaned samples with at least one recovered edge.
# They are NOT used for final feature selection.

GRAPH_FEATURES = [
    'n_transfers',
    'n_endpoints',
    'n_edges',
    'n_senders',
    'n_receivers',
    'density',
    'avg_degree',
    'top_sender_share',
    'amount_sum',
    'amount_median',
    'first_t_frac',
    'mean_t_frac'
]

GRAPH_DIAGNOSTIC_ONLY = [
    'amount_sum',
    'amount_median',
    'top_sender_share'
]

GRAPH_MODEL_FEATURES = [
    c for c in GRAPH_FEATURES
    if c not in GRAPH_DIAGNOSTIC_ONLY
]

graph_auc = pd.DataFrame({
    q: {
        c: feature_auc(gfeat[q], c)
        for c in GRAPH_FEATURES
    }
    for q in EXPERIMENTS
})

graph_n = pd.DataFrame({
    q: {
        c: gfeat[q][[c, 'rug_pull_label']]
        .dropna()
        .shape[0]
        for c in GRAPH_FEATURES
    }
    for q in EXPERIMENTS
})

pd.concat(
    {
        'auc': graph_auc.round(3),
        'n': graph_n
    },
    axis=1
)

In [ ]:
# Descriptive graph-feature association for q1.
# This plot is EDA only and is not used to select the final feature set.

fig, ax = plt.subplots(figsize=(7, 4))

(graph_auc.loc[GRAPH_MODEL_FEATURES, 'q1'] - 0.5).sort_values().plot(kind='barh', ax=ax, color='#4c72b0')

ax.axvline(0, color='k', lw=0.8)
ax.set_xlabel('AUC - 0.5   (positive = higher feature values in rug pulls)')
ax.set_title(f'q1 graph features — descriptive association (n={len(gfeat["q1"])})')

plt.tight_layout()
plt.show()

**<span style="color:red">Observation:</span>** Within q1 samples that contain at least
one recovered edge, the figure summarizes the univariate rank association between each
candidate graph feature and the rug-pull label. An AUC above 0.5 indicates that larger
feature values tend to occur for rug-pull samples, whereas an AUC below 0.5 indicates
the opposite direction.

This analysis is descriptive only. It is not used to select the final graph predictors,
because it is computed on the complete cleaned q1 graph subset rather than on the
training partition created later in this notebook. Training-only model-based feature
importance is computed only after the central chronological split has been fixed.

Recovered-edge presence is also class-dependent, and the recovered graphs are partial
mint-visible representations, so these associations are conditional on the non-empty
recovered-graph subset.

## Step 5 — Full Recovered Transfer Multigraph Example

The visualization below shows one token-window sample as a full recovered directed
transfer multigraph. Every retained recovered transfer row is drawn individually.

No node filtering, edge aggregation, or topology simplification is
used in the figure itself. If multiple retained transfers occur between the same ordered
pair of endpoints, they are drawn as separate directed edges rather than being collapsed
into a single edge.

Each node is a recovered transfer endpoint. The figure should still be interpreted as a
partial mint-visible recovered transfer graph, not as a complete history of all token
transfers on-chain. The graph is complete only with respect to the retained recovered
edge rows for the selected token-window sample.

In [ ]:
def build_recovered_multigraph(edges_df, token_mint):
    """
    Build a full recovered transfer multigraph for one token-window sample.

    Every retained recovered transfer row is kept as a separate directed edge.
    No aggregation across repeated source-target pairs is performed.
    """
    sub = edges_df[edges_df['token_mint'] == token_mint].copy().reset_index(drop=True)

    G = nx.MultiDiGraph()

    for row_id, row in sub.iterrows():
        G.add_edge(
            row['source_address'],
            row['target_address'],
            key=row_id,
            tx_signature=row['tx_signature'],
            amount=row['amount'],
            t_frac=row['t_frac'],
            observation_start=row['observation_start'],
            observation_end=row['observation_end']
        )

    return G, sub

In [ ]:
def graph_structure_diagnostics(edges_df, graph_feature_df):
    """
    Structural diagnostics of the simple directed topology induced by the
    retained recovered edge rows.

    Repeated source-target rows are collapsed only for these topology
    diagnostics. The original recovered multigraph is not modified.
    """

    model_tokens = set(graph_feature_df['token_mint'])

    label_lookup = (graph_feature_df.set_index('token_mint')['rug_pull_label'].to_dict())

    # Structural diagnostics depend only on distinct directed endpoint pairs.
    # Deduplicate once globally instead of rescanning the full edge table
    # separately for every token.
    pair_rows = (
        edges_df.loc[
            edges_df['token_mint'].isin(model_tokens),
            [
                'token_mint',
                'source_address',
                'target_address'
            ]
        ]
        .drop_duplicates()
    )

    rows = []
    seen_tokens = set()

    for token, sub in pair_rows.groupby('token_mint', sort=False):
        seen_tokens.add(token)

        G = nx.from_pandas_edgelist(
            sub,
            source='source_address',
            target='target_address',
            create_using=nx.DiGraph()
        )

        n_nodes = G.number_of_nodes()
        n_edges = G.number_of_edges()

        assert n_nodes > 0
        assert n_edges > 0

        distinct_neighbor_counts = np.array(
            [
                len(
                    set(G.predecessors(node))
                    | set(G.successors(node))
                )
                for node in G.nodes()
            ],
            dtype=float
        )

        topological_leaf_fraction = (
            (distinct_neighbor_counts == 1).mean()
            if n_nodes > 1
            else np.nan
        )

        multi_neighbor_fraction = (
            (distinct_neighbor_counts >= 2).mean()
            if n_nodes > 0
            else np.nan
        )

        max_distinct_neighbors = (
            distinct_neighbor_counts.max()
            if n_nodes > 0
            else np.nan
        )

        distinct_neighbor_dominance = (
            max_distinct_neighbors / (n_nodes - 1)
            if n_nodes > 1
            else np.nan
        )

        U = G.to_undirected()

        n_components = nx.number_connected_components(U)

        reciprocity = nx.reciprocity(G)

        rows.append(
            {
                'token_mint': token,
                'rug_pull_label': label_lookup[token],
                'n_endpoints': n_nodes,
                'n_edges': n_edges,
                'n_components': n_components,
                'max_distinct_neighbors': max_distinct_neighbors,
                'topological_leaf_fraction': topological_leaf_fraction,
                'distinct_neighbor_dominance': distinct_neighbor_dominance,
                'multi_neighbor_fraction': multi_neighbor_fraction,
                'reciprocity': reciprocity
            }
        )

    missing_tokens = model_tokens - seen_tokens

    assert not missing_tokens, (f'Graph-feature tokens missing from structural diagnostics: {len(missing_tokens)}')

    result = (
        pd.DataFrame(rows)
        .sort_values(
            'token_mint',
            kind='mergesort'
        )
        .reset_index(drop=True)
    )

    assert len(result) == len(graph_feature_df)

    return result

In [ ]:
structure_by_q = {
    q: graph_structure_diagnostics(
        edges_q[q],
        gfeat[q]
    )
    for q in EXPERIMENTS
}

structural_summary = pd.DataFrame({
    q: {
        'recovered graphs':
            len(structure_by_q[q]),

        'median topological leaf fraction':
            structure_by_q[q]['topological_leaf_fraction'].median(),

        'median distinct-neighbor dominance':
            structure_by_q[q]['distinct_neighbor_dominance'].median(),

        'median reciprocity':
            structure_by_q[q]['reciprocity'].median()
    }
    for q in EXPERIMENTS
}).T

display(structural_summary.round(3))

**<span style="color:red">Observation:</span>** The structural summary above characterizes
the recovered non-empty graphs at each observation window. The statistics are descriptive
properties of the partial recovered graphs and should not be interpreted as complete
token-wide network statistics.

The example selected below is intentionally chosen for readability and structural richness,
not because it is representative of either outcome class.

In [ ]:
VIS_Q = 'q3'

visual_candidates = structure_by_q[VIS_Q].copy()

visual_candidates = visual_candidates[
    (visual_candidates['n_components'] == 1)
    &
    (visual_candidates['n_endpoints'].between(50, 200))
].copy()

assert not visual_candidates.empty

leaf_cut = visual_candidates['topological_leaf_fraction'].quantile(0.25)

dominance_cut = visual_candidates['distinct_neighbor_dominance'].quantile(0.25)

multi_neighbor_cut = visual_candidates['multi_neighbor_fraction'].quantile(0.75)

rich_candidates = visual_candidates[
    (visual_candidates['topological_leaf_fraction'] <= leaf_cut)
    &
    (visual_candidates['distinct_neighbor_dominance'] <= dominance_cut)
    &
    (visual_candidates['multi_neighbor_fraction'] >= multi_neighbor_cut)
].copy()

if rich_candidates.empty:
    rich_candidates = visual_candidates.copy()

selected_graph_row = (
    rich_candidates
    .sort_values(
        [
            'n_edges',
            'n_endpoints',
            'token_mint'
        ],
        ascending=[
            False,
            False,
            True
        ]
    )
    .iloc[0]
)

display(
    selected_graph_row[
        [
            'token_mint',
            'n_endpoints',
            'n_edges',
            'n_components',
            'topological_leaf_fraction',
            'multi_neighbor_fraction',
            'distinct_neighbor_dominance',
            'reciprocity'
        ]
    ]
    .to_frame('value')
)

In [ ]:
selected_token = selected_graph_row['token_mint']

G_selected, selected_edges = build_recovered_multigraph(edges_q[VIS_Q], selected_token)

distinct_directed_pairs = selected_edges[['source_address', 'target_address']].drop_duplicates().shape[0]

print('Observation window:', VIS_Q)
print('Token mint:', selected_token)
print('Retained recovered transfer rows:', G_selected.number_of_edges())
print('Transfer endpoints:', G_selected.number_of_nodes())
print('Distinct directed endpoint pairs:', distinct_directed_pairs)

In [ ]:
os.makedirs('./artifacts/figures', exist_ok=True)

pos = nx.spring_layout(
    G_selected,
    seed=42,
    iterations=1000
)

fig, ax = plt.subplots(figsize=(11, 9))

# Draw all nodes with one style only.
nx.draw_networkx_nodes(
    G_selected,
    pos,
    node_size=85,
    node_shape='o',
    alpha=0.85,
    ax=ax
)

# Draw every retained transfer row separately.
# If multiple transfers exist between the same ordered pair,
# assign different curvatures so that they remain individually visible.
pair_to_keys = {}

for u, v, k in G_selected.edges(keys=True):
    pair_to_keys.setdefault((u, v), []).append(k)

for (u, v), keys in pair_to_keys.items():
    m = len(keys)

    if m == 1:
        radii = [0.0]
    else:
        max_rad = min(0.6, 0.08 * (m - 1))
        radii = np.linspace(-max_rad, max_rad, m)

    for k, rad in zip(keys, radii):
        nx.draw_networkx_edges(
            G_selected,
            pos,
            edgelist=[(u, v, k)],
            width=0.8,
            alpha=0.35,
            arrows=True,
            arrowsize=7,
            arrowstyle='-|>',
            connectionstyle=f'arc3,rad={rad}',
            ax=ax
        )

ax.set_title(
    'Full recovered transfer multigraph\n'
    f'{G_selected.number_of_nodes()} endpoints, '
    f'{G_selected.number_of_edges()} retained transfer rows, '
    f'{distinct_directed_pairs} distinct directed endpoint pairs'
)

ax.axis('off')
plt.tight_layout()

fig.savefig('./artifacts/figures/example_recovered_transfer_multigraph.pdf', bbox_inches='tight')
fig.savefig('./artifacts/figures/example_recovered_transfer_multigraph.png', dpi=600, bbox_inches='tight')

plt.show()

**<span style="color:red">Observation:</span>** The figure shows the selected token-window
sample as a full recovered directed transfer multigraph. Every retained recovered transfer 
row is drawn as a separate directed edge, so repeated transfers between the same ordered pair of
endpoints appear as separate directed edges.

No node filtering or edge aggregation is used in the figure
itself. The layout is used only to position nodes on the page. The visualization should
therefore be interpreted as a direct view of the retained recovered edge rows for the
selected sample.

The graph is still a partial mint-visible recovered transfer graph rather than a complete
on-chain transfer history, because the edge collector starts from mint-address-based
transaction discovery.

# Leakage and Data-Validity Checks

## Step 1 — Verify Label Construction Consistency

In [ ]:
valid = labels['label_status'] == 'VALID'
non_valid = ~valid

expected_windows = {
    'q1': 750,
    'q2': 2127,
    'q3': 5585
}

label_key_cols = ['lp_address', 'token_mint', 'experiment_id']
sample_key_cols = ['lp_address', 'token_mint']

sample_experiment_counts = labels.groupby(sample_key_cols)['experiment_id'].nunique()

expected_rug = labels['liquidity_drop_flag'].eq(True) & labels['price_drop_flag'].eq(True) & labels['persistent_liquidity_depletion_flag'].eq(True)
valid_rug = valid & labels['rug_pull_label'].eq(1)

In [ ]:
label_checks = {}

label_checks['label key is unique'] = ~labels.duplicated(label_key_cols).any()

label_checks['only expected experiments are present'] = set(labels['experiment_id'].dropna().unique()) == set(EXPERIMENTS)

label_checks['2700 rows per experiment'] = labels.groupby('experiment_id').size().reindex(EXPERIMENTS).eq(2700).all()

label_checks['2700 unique token-LP samples'] = labels[sample_key_cols].drop_duplicates().shape[0] == 2700

label_checks['each token-LP sample has q1/q2/q3'] = sample_experiment_counts.eq(len(EXPERIMENTS)).all()

label_checks['observation end matches window'] = ((labels['observation_end_time'] - labels['token_creation_time']) == labels['observation_window_seconds']).all()

label_checks['24h outcome horizon'] = ((labels['outcome_end_time'] - labels['token_creation_time']) == 86400).all()

label_checks['q/window mapping'] = all((labels.loc[labels['experiment_id'] == q, 'observation_window_seconds'] == seconds).all()
    for q, seconds in expected_windows.items()
)

label_checks['VALID labels are binary'] = labels.loc[valid, 'rug_pull_label'].isin([0, 1]).all()

label_checks['non-VALID labels are unset'] = labels.loc[non_valid, 'rug_pull_label'].isna().all()

label_checks['VALID labels available after cutoff'] = (labels.loc[valid, 'label_available_time'] > labels.loc[valid, 'observation_end_time']).all()

label_checks['VALID labels available within outcome horizon'] = (labels.loc[valid, 'label_available_time'] <= labels.loc[valid, 'outcome_end_time']).all()

label_checks['binary label matches three-rule conjunction'] = (labels.loc[valid, 'rug_pull_label'].astype(int) == expected_rug.loc[valid].astype(int)).all()

label_checks['rug liquidity drop exceeds 80%'] = (labels.loc[valid_rug, 'liquidity_drop_ratio'] > 0.80).all()

label_checks['rug price drop exceeds 90%'] = (labels.loc[valid_rug, 'price_drop_ratio'] > 0.90).all()

label_checks['rug persistence ratio is at least 80%'] = (labels.loc[valid_rug, 'persistence_ratio'] >= 0.8).all()

label_checks['rug persistence endpoint remains below threshold'] = (labels.loc[valid_rug, 'liquidity_at_persistence_end'] <= labels.loc[valid_rug, 'persistence_liquidity_threshold']).all()

label_checks['30-minute persistence confirmation'] = (labels['persistence_confirmation_seconds'] == 1800).all()

label_checks['rug evidence is strictly after the observation window'] = all((labels.loc[valid_rug, col] > labels.loc[valid_rug, 'observation_end_time']).all()
    for col in [
        'liquidity_drop_factor_time',
        'price_drop_factor_time',
        'persistent_liquidity_depletion_factor_time'
    ]
)

label_checks['VALID rug liquidity event clears 1-minute boundary ambiguity'] = ((labels.loc[valid_rug, 'liquidity_drop_factor_time'] - 60) > labels.loc[valid_rug, 'observation_end_time']).all()

label_checks['VALID labels become available at the 24h outcome endpoint'] = (labels.loc[ valid, 'label_available_time'] == labels.loc[valid, 'outcome_end_time']).all()

for name, passed in label_checks.items():
    print(f'{"OK  " if passed else "FAIL"} {name}')

if not all(label_checks.values()):
    raise AssertionError('Label construction consistency check failed')

## Step 2 — Check Direct Predictor/Outcome Column Overlap

In [ ]:
PREDICTOR_COLUMNS = set(NODE_MODEL_FEATURES + GRAPH_MODEL_FEATURES)

LABEL_AUDIT_COLUMNS = set(labels.columns) - {
    'lp_address',
    'token_mint',
    'experiment_id',
    'observation_window_seconds'
}

leaking_predictors = sorted(PREDICTOR_COLUMNS & LABEL_AUDIT_COLUMNS)

print('predictor columns overlapping label-audit columns:', leaking_predictors)

assert not leaking_predictors, f'Potential direct leakage: {leaking_predictors}'

## Step 3 — Verify Label Availability After the Observation Window

In [ ]:
for q in EXPERIMENTS:
    invalid = (
        nodes_q[q]['label_lead_time'].isna()
        | (nodes_q[q]['label_lead_time'] <= 0)
    )

    print(f'{q}: invalid label timing = {invalid.sum()}')

    assert invalid.sum() == 0

## Step 4 — Predictor Provenance Boundary

| Predictor group | Verified collector behavior | Remaining limitation |
|---|---|---|
| Numerical token-level features | Trade records are locally restricted to the half-open `[observation_start, observation_end)` interval. Incomplete/unparseable trade histories do not produce numerical features. Liquidity at the cutoff uses a backward snapshot, and temporally distant snapshots remain missing rather than using current/post-cutoff values. | Historical API availability and missingness remain external data-quality constraints. |
| Initial authority metadata | `mint_authority` and `freeze_authority` are parsed from the selected mint-creation transaction rather than queried from later token state. | A missing field cannot currently distinguish a genuinely absent authority from incomplete recovery of initialization metadata; authority indicators are therefore EDA-only. |
| Creator metadata | `creators` contains non-mint signer accounts from the selected mint-creation transaction and is excluded from the modelling whitelist. | Empty recovered metadata does not prove that no creator-side account existed; the fallback creation transaction may also be approximate. |
| Graph topology/timing | Edge jobs use the label-defined half-open observation window, and recovered transfer timestamps are checked again during processing. | Transfer discovery begins with `getSignaturesForAddress(token_mint)`, so transfers not visible through the mint-address history may be missed. |
| Transfer amounts | Amounts are preserved rather than discarding transfers when decimal metadata is unavailable. | Values may mix decimal-scaled token units and raw base units. `amount_sum`, `amount_median`, and `top_sender_share` are therefore diagnostic-only until amount provenance is explicitly represented. |

The collector audit supports cutoff-safe construction of the main historical Numerical token-level features
 and edge timestamps. The remaining limitations concern data availability,
recovery completeness, ambiguous metadata provenance, and graph-discovery coverage rather
than identified direct future-time leakage.

## Step 5 — Experimental Validity Notes

- `IN_WINDOW_EVENT` samples are excluded from supervised model-ready data because the
  rug-pull event occurs inside the observation window rather than after prediction time.

- The `full` scope is the primary dataset for each observation window. The `panel` scope is
  used only as a controlled secondary analysis with the same tokens across q1/q2/q3.

- A single central chronological train/validation/test assignment is created only after
  all complete-data EDA and leakage audits are finished. The assignment is defined once
  at the token-mint level and reused across q1/q2/q3 and across tabular and graph-model
  representations. Class labels are not used to allocate samples to partitions.

- Learned missing-data preprocessing and model-based feature importance are fitted only
  after the central chronological split has been fixed. The median imputer is fitted
  using training predictors only and is applied unchanged to validation and test data.
  Model-based feature importance is also computed using training data only. These fitted
  preprocessing objects, final data variants, and training-only importance results are
  saved by this notebook and are not recomputed downstream.

- `graph_edges.csv` contains only retained recovered-edge rows. Absence of an edge row
  does not by itself prove that the token had no transfers.

- Recovered transaction graphs represent transfer activity visible through the
  mint-address transaction history and should not be interpreted as complete token
  transaction histories.

- Token-level feature missingness is preserved throughout the complete-data EDA.
  Imputation is performed only after splitting. Incomplete trade histories do not produce
  numerical trade features, undefined percentage changes remain missing in the raw
  representation, and unavailable or temporally distant liquidity snapshots are not
  replaced with current or post-cutoff values.

- Transfer `amount` values may use decimal-scaled token units or raw base units depending
  on available transaction metadata. Amount-based graph features, including
  `top_sender_share`, remain diagnostic only.

- One token has VALID labels for all three observation windows but no corresponding node
  rows in the current output. Missing numerical features alone do not suppress node rows
  under the collector logic, so the absence concerns the persisted node output rather
  than ordinary feature missingness.

- Recovered-edge presence is class-dependent, so the subset of samples with non-empty
  recovered graphs should not be treated as representative of the full supervised
  dataset.

- Because the edge table does not contain instruction-level event identifiers, distinct
  transfers within the same transaction that share the same source, target, and amount
  cannot always be distinguished from duplicated collection rows by the current
  deduplication key.

- Supervised evaluation is conditional on label availability. `UNKNOWN_*` outcomes are
  retained in the audit table but excluded from binary modelling.

# Final Artifact Preparation

The notebook first saves the complete cleaned, unsplit artifacts with missing predictor
values preserved.

It then creates the central chronological train/validation/test assignment, applies the
temporal embargo audit, generates both predefined missing-data representations,
computes training-only model-based feature importance, and exports the final
model-ready artifacts.

No model-development preprocessing is fitted before the chronological split.

In [ ]:
os.makedirs('./artifacts', exist_ok=True)
os.makedirs('./artifacts/label_audit', exist_ok=True)
os.makedirs('./artifacts/figures', exist_ok=True)

In [ ]:
sample_manifest = pd.concat(
    [
        nodes_q[q][
            [
                'lp_address',
                'token_mint',
                'creation_time',
                'observation_start',
                'observation_end',
                'label_available_time',
                'rug_pull_label',
                'has_recovered_edges'
            ]
        ]
        .assign(experiment_id=q)
        for q in EXPERIMENTS
    ],
    ignore_index=True
)

manifest_key = [
    'lp_address',
    'token_mint',
    'experiment_id'
]

assert not sample_manifest.duplicated(manifest_key).any()

assert 'split' not in sample_manifest.columns

sample_manifest.to_parquet('./artifacts/sample_manifest.parquet', index=False)

print('sample manifest rows:', len(sample_manifest))

display(
    sample_manifest
    .groupby('experiment_id')
    .agg(
        samples=('token_mint', 'size'),
        tokens=('token_mint', 'nunique'),
        rug_rate=('rug_pull_label', 'mean'),
        recovered_graphs=('has_recovered_edges', 'sum')
    )
)

In [ ]:
for q in EXPERIMENTS:
    labels_q[q].to_parquet(f'./artifacts/label_audit/{q}_labels.parquet', index=False)

label_node_audit.to_parquet('./artifacts/label_node_audit.parquet', index=False)

In [ ]:
nodes.to_parquet('./artifacts/cleaned_nodes.parquet', index=False)
edges.to_parquet('./artifacts/cleaned_edges.parquet', index=False)
labels.to_parquet('./artifacts/cleaned_labels.parquet', index=False)

print('Saved cleaned root tables:')
print('  cleaned_nodes.parquet')
print('  cleaned_edges.parquet')
print('  cleaned_labels.parquet')

In [ ]:
feature_registry = {
    'node_model_candidates': NODE_MODEL_FEATURES,
    'graph_model_candidates': GRAPH_MODEL_FEATURES,
    'graph_diagnostic_only': GRAPH_DIAGNOSTIC_ONLY,

    'split_policy': {
        'unit': 'token_mint',
        'type': 'chronological_with_boundary_embargo',
        'target_train_fraction': 0.70,
        'target_validation_fraction': 0.15,
        'target_test_fraction': 0.15,
        'ordering_variable': 'creation_time',
        'allocation_uses_label': False,
        'same_assignment_across_q_windows': True,
        'same_assignment_across_model_branches': True,
        'embargo_rule': (
            'An earlier-partition token is excluded from model development when its '
            'label_available_time is not strictly earlier than the creation time at '
            'which the next chronological partition begins.'
        )
    },

    'feature_importance_policy': (
        'Model-based feature importance is implemented and executed directly in this '
        'notebook after the central chronological split is fixed. It is computed '
        'separately for q1/q2/q3 using training rows from the raw_nan representation '
        'only. Validation and test rows do not enter estimator fitting or permutation '
        'scoring. The resulting rankings are saved as final experimental artifacts '
        'and are not recomputed downstream.'
    ),

    'missing_data_variants': {
        'raw_nan': (
            'Preserve missing predictor values without imputation.'
        ),
        'median_indicator': (
            'Median imputation fitted on training predictors only, with explicit '
            'missingness indicators. The fitted imputer is applied unchanged to '
            'validation and test data.'
        )
    },

    'missing_data_policy': (
        'Missing-data representations are generated in this notebook only after the '
        'chronological split is fixed. Labels are never used as imputation inputs.'
    )
}

with open('./artifacts/feature_registry.json', 'w', encoding='utf-8') as f:
    json.dump(feature_registry, f, indent=2)

print(json.dumps(feature_registry, indent=2))

In [ ]:
for scope in ['full', 'panel']:

    for q in EXPERIMENTS:
        folder = f'./artifacts/{scope}/{q}'

        os.makedirs(folder, exist_ok=True)

        # Full cleaned token-level table.
        nodes_out = (
            apply_scope(nodes_q[q], scope)
            .drop(
                columns=[
                    'n_transfers',
                    'has_recovered_edges'
                ],
                errors='ignore'
            )
        )

        assert 'split' not in nodes_out.columns

        nodes_out.to_parquet(f'{folder}/nodes.parquet', index=False)

        # Restricted candidate table for tabular modelling.
        model_nodes_out = nodes_out[
            [
                'lp_address',
                'token_mint',
                'creation_time',
                'rug_pull_label'
            ]
            + NODE_MODEL_FEATURES
        ].copy()

        model_nodes_out.to_parquet(f'{folder}/model_nodes.parquet', index=False)

        # Clean recovered transfer rows for the same model-ready
        # token population represented by nodes_out.
        model_keys = nodes_out[['lp_address', 'token_mint']].drop_duplicates()

        edges_out = edges_q[q].merge(model_keys, on=['lp_address', 'token_mint'], how='inner', validate='many_to_one')

        edge_keys_out = pd.MultiIndex.from_frame(edges_out[['lp_address', 'token_mint']].drop_duplicates())

        model_keys_index = pd.MultiIndex.from_frame(model_keys)

        assert edge_keys_out.isin(model_keys_index).all()

        edges_out.to_parquet(f'{folder}/edges.parquet', index=False)

        # Graph-summary table: only samples with a non-empty
        # recovered graph are present by construction.
        graph_out = apply_scope(gfeat[q], scope)

        assert 'split' not in graph_out.columns

        graph_out.to_parquet(f'{folder}/graph_features.parquet', index=False)

        # Restricted candidate graph-summary predictors.
        model_graph_out = graph_out[
            [
                'lp_address',
                'token_mint',
                'creation_time',
                'rug_pull_label'
            ]
            + GRAPH_MODEL_FEATURES
        ].copy()

        model_graph_out.to_parquet(f'{folder}/model_graph_features.parquet', index=False)

        # Structural diagnostic table.
        structure_out = apply_scope(structure_by_q[q], scope)

        structure_out.to_parquet(f'{folder}/graph_structure.parquet', index=False)

    print(f'saved scope={scope}')

# Chronological Split and Leakage-Safe Experimental Preparation

All complete-data descriptive analyses and leakage audits are complete above.

The following cells create the final model-development partitions and preprocessing
artifacts. One chronological assignment is defined at the unique token-mint level and is
reused across q1/q2/q3 and across tabular and graph-model representations.

The target proportions are 70% training, 15% validation, and 15% test. Train/validation/test boundaries are determined from creation time only and are not
stratified by the outcome label. `label_available_time` is used only for the fixed
boundary purge/embargo check; the class value itself does not affect partition
assignment.

Because the operational label uses future outcome information, samples immediately before
the validation and test periods are embargoed whenever their labels would not yet be
available when the following period begins.

In [ ]:
# ============================================================
# CENTRAL CHRONOLOGICAL SPLIT
# ============================================================

TRAIN_TARGET = 0.70
VALIDATION_TARGET = 0.15
TEST_TARGET = 0.15

assert np.isclose(TRAIN_TARGET + VALIDATION_TARGET + TEST_TARGET, 1.0)

os.makedirs('./artifacts/splits', exist_ok=True)

# One central record per unique token.
split_base = sample_manifest[
    [
        'token_mint',
        'creation_time',
        'label_available_time'
    ]
].copy()

for col in ['creation_time', 'label_available_time']:
    split_base[col] = pd.to_numeric(split_base[col], errors='raise')

assert split_base['token_mint'].notna().all()
assert split_base['creation_time'].notna().all()
assert split_base['label_available_time'].notna().all()

# Creation time must agree across q1/q2/q3 for the same token.
creation_nunique = split_base.groupby('token_mint')['creation_time'].nunique()

assert (creation_nunique == 1).all()

label_availability_nunique = split_base.groupby('token_mint')['label_available_time'].nunique()

assert (label_availability_nunique == 1).all()

# label_available_time is identical across q1/q2/q3 for each
# token, as verified above, so one common value is retained.
token_split = (
    split_base
    .groupby('token_mint', as_index=False)
    .agg(
        creation_time=('creation_time', 'first'),
        label_available_time=('label_available_time', 'first')
    )
    .sort_values(
        ['creation_time', 'token_mint'],
        kind='mergesort'
    )
    .reset_index(drop=True)
)

n_tokens = len(token_split)

assert n_tokens > 0


# ------------------------------------------------------------
# Choose chronological boundaries while keeping identical
# creation timestamps together.
# ------------------------------------------------------------

time_counts = token_split.groupby('creation_time', sort=True).size()

counts_before_time = time_counts.cumsum().shift(1, fill_value=0)

target_train_n = TRAIN_TARGET * n_tokens
target_train_validation_n = (TRAIN_TARGET + VALIDATION_TARGET) * n_tokens

# Validation begins at the timestamp whose number of earlier
# tokens is closest to the 70% target.
validation_start = counts_before_time.sub(target_train_n).abs().idxmin()

# Test boundary must be strictly later than validation boundary.
test_candidates = counts_before_time[counts_before_time.index > validation_start]

test_start = test_candidates.sub(target_train_validation_n).abs().idxmin()

assert validation_start < test_start


# ------------------------------------------------------------
# Raw chronological periods
# ------------------------------------------------------------

creation = token_split['creation_time']
label_available = token_split['label_available_time']

train_period = creation < validation_start

validation_period = (creation >= validation_start) & (creation < test_start)

test_period = creation >= test_start


# ------------------------------------------------------------
# Boundary embargo
# ------------------------------------------------------------

train_ok = train_period & (label_available < validation_start)

train_validation_embargo = train_period & ~train_ok

validation_ok = validation_period & (label_available < test_start)

validation_test_embargo = validation_period & ~validation_ok

token_split['split'] = pd.Series(pd.NA, index=token_split.index, dtype='object')

token_split['embargo_boundary'] = pd.Series(pd.NA, index=token_split.index, dtype='object')

token_split.loc[train_ok, 'split'] = 'train'

token_split.loc[train_validation_embargo, 'split'] = 'embargo'

token_split.loc[train_validation_embargo, 'embargo_boundary'] = 'train_validation'

token_split.loc[validation_ok, 'split'] = 'validation'

token_split.loc[validation_test_embargo, 'split'] = 'embargo'

token_split.loc[validation_test_embargo, 'embargo_boundary'] = 'validation_test'

token_split.loc[test_period, 'split'] = 'test'

assert token_split['split'].notna().all()
assert not token_split['token_mint'].duplicated().any()


# Human-readable UTC timestamps.
token_split['creation_time_utc'] = pd.to_datetime(token_split['creation_time'], unit='s', utc=True)

token_split['label_available_time_utc'] = pd.to_datetime(token_split['label_available_time'], unit='s', utc=True)

validation_start_utc = pd.to_datetime(validation_start, unit='s', utc=True)

test_start_utc = pd.to_datetime(test_start, unit='s', utc=True)

print('Unique model-ready tokens:', n_tokens)
print('Target proportions: 70% / 15% / 15%')
print('Validation period begins:', validation_start_utc)
print('Test period begins:', test_start_utc)

display(token_split['split'].value_counts().rename_axis('split').to_frame('tokens'))

In [ ]:
# ============================================================
# SPLIT AUDIT
# ============================================================

model_token_split = token_split[
    token_split['split'].isin(
        ['train', 'validation', 'test']
    )
].copy()

train_tokens = model_token_split[model_token_split['split'] == 'train']

validation_tokens = model_token_split[model_token_split['split'] == 'validation']

test_tokens = model_token_split[model_token_split['split'] == 'test']

assert len(train_tokens) > 0
assert len(validation_tokens) > 0
assert len(test_tokens) > 0

# Strict chronological ordering.
assert train_tokens['creation_time'].max() < validation_tokens['creation_time'].min()

assert validation_tokens['creation_time'].max() < test_tokens['creation_time'].min()

# Outcome availability must not cross the following boundary.
assert train_tokens['label_available_time'].max() < validation_tokens['creation_time'].min()

assert validation_tokens['label_available_time'].max() < test_tokens['creation_time'].min()


# ------------------------------------------------------------
# Attach the central assignment to all q-window samples.
# ------------------------------------------------------------

sample_manifest_with_split = sample_manifest.merge(
    token_split[
        [
            'token_mint',
            'split',
            'embargo_boundary'
        ]
    ],
    on='token_mint',
    how='left',
    validate='many_to_one'
)

assert sample_manifest_with_split['split'].notna().all()

# Same token -> same assignment everywhere.
assert sample_manifest_with_split.groupby('token_mint')['split'].nunique().max() == 1


# ------------------------------------------------------------
# Descriptive split audit.
# LABEL IS USED HERE ONLY AFTER ALLOCATION.
# ------------------------------------------------------------

split_summary = (
    sample_manifest_with_split
    .groupby(
        ['experiment_id', 'split'],
        observed=True
    )
    .agg(
        samples=('token_mint', 'size'),
        rugs=('rug_pull_label', 'sum'),
        rug_rate=('rug_pull_label', 'mean'),
        recovered_graphs=('has_recovered_edges', 'sum'),
        first_creation_time=('creation_time', 'min'),
        last_creation_time=('creation_time', 'max')
    )
    .reset_index()
)

split_summary['non_rugs'] = split_summary['samples'] - split_summary['rugs']

split_summary['rug_rate_percent'] = split_summary['rug_rate'] * 100.0

split_summary['first_creation_utc'] = pd.to_datetime(split_summary['first_creation_time'], unit='s', utc=True)

split_summary['last_creation_utc'] = pd.to_datetime(split_summary['last_creation_time'], unit='s', utc=True)

split_summary = split_summary[
    [
        'experiment_id',
        'split',
        'samples',
        'non_rugs',
        'rugs',
        'rug_rate_percent',
        'recovered_graphs',
        'first_creation_utc',
        'last_creation_utc'
    ]
]


# Ensure every non-embargo partition has both classes for every q.
for q in EXPERIMENTS:
    q_model = sample_manifest_with_split[
        (sample_manifest_with_split['experiment_id'] == q)
        & sample_manifest_with_split['split'].isin(
            ['train', 'validation', 'test']
        )
    ]

    for split_name in ['train', 'validation', 'test']:
        classes = set(
            q_model.loc[
                q_model['split'] == split_name,
                'rug_pull_label'
            ].dropna().astype(int).unique()
        )

        assert classes == {0, 1}, (f'{q}/{split_name} does not contain both classes: {classes}')


# ------------------------------------------------------------
# Save central split source of truth.
# ------------------------------------------------------------

token_split.to_parquet('./artifacts/splits/chronological_split_manifest.parquet', index=False)

token_split.to_csv('./artifacts/splits/chronological_split_manifest.csv', index=False)

sample_manifest_with_split.to_parquet('./artifacts/splits/sample_manifest_with_split.parquet', index=False)

split_summary.to_csv('./artifacts/splits/split_summary_by_experiment.csv', index=False)


split_config = {
    'unit': 'token_mint',
    'ordering_variable': 'creation_time',
    'target_train_fraction': TRAIN_TARGET,
    'target_validation_fraction': VALIDATION_TARGET,
    'target_test_fraction': TEST_TARGET,
    'allocation_uses_label': False,

    'validation_start_unix': int(validation_start),
    'validation_start_utc': validation_start_utc.isoformat(),

    'test_start_unix': int(test_start),
    'test_start_utc': test_start_utc.isoformat(),

    'purge_rule': (
        'An earlier-partition token is embargoed when '
        'label_available_time is not strictly earlier than '
        'the creation time at which the next partition begins.'
    ),

    'unique_tokens_total': int(n_tokens),

    'train_tokens': int((token_split['split'] == 'train').sum()),

    'validation_tokens': int((token_split['split'] == 'validation').sum()),

    'test_tokens': int((token_split['split'] == 'test').sum()),

    'embargo_tokens': int((token_split['split'] == 'embargo').sum()),
}

with open('./artifacts/splits/split_config.json', 'w', encoding='utf-8') as f:
    json.dump(split_config, f, indent=2)

print(json.dumps(split_config, indent=2))

display(split_summary)

# Leakage-Safe Missing-Data Variants

The central split is now fixed.

Exactly two final tabular representations are created separately for q1, q2, and q3.
The raw representation preserves NaN values. Median imputation is fitted on training
predictors only and then applied unchanged to validation and test rows.

For the median-imputed representation, an explicit missingness indicator is retained
for every original predictor. The indicators are computed from the original
pre-imputation values.

The outcome label is never supplied to the imputation model.

In [ ]:
# ============================================================
# MISSING-DATA PREPROCESSING HELPERS
# ============================================================

MISSING_INDICATOR_COLUMNS = [f'{col}__missing' for col in NODE_MODEL_FEATURES]

def numeric_predictors(df):
    X = df[NODE_MODEL_FEATURES].copy()

    for col in NODE_MODEL_FEATURES:
        X[col] = pd.to_numeric(X[col], errors='raise')

    infinite_cols = [
        col
        for col in NODE_MODEL_FEATURES
        if np.isinf(X[col].dropna().to_numpy(dtype=float)).any()
    ]

    if infinite_cols:
        raise ValueError(f'Infinite predictor values found in: {infinite_cols}')

    return X


def add_missing_indicators(output_df, original_X):
    result = output_df.copy()

    for col in NODE_MODEL_FEATURES:
        result[f'{col}__missing'] = original_X[col].isna().astype('int8').to_numpy()

    return result


def rebuild_imputed_dataset(base_df, transformed_X):
    id_cols = [
        'lp_address',
        'token_mint',
        'experiment_id',
        'creation_time',
        'rug_pull_label',
        'split'
    ]

    result = base_df[id_cols].copy().reset_index(drop=True)

    transformed_df = pd.DataFrame(transformed_X, columns=NODE_MODEL_FEATURES)

    result = pd.concat([result, transformed_df], axis=1)

    original_X = numeric_predictors(base_df).reset_index(drop=True)

    result = add_missing_indicators(result, original_X)

    return result


def audit_imputed_variant(raw_df, processed_df, variant_name):
    assert len(raw_df) == len(processed_df)

    for col in [
        'lp_address',
        'token_mint',
        'experiment_id',
        'creation_time',
        'rug_pull_label',
        'split'
    ]:
        assert raw_df[col].reset_index(drop=True).equals(processed_df[col].reset_index(drop=True))

    # Imputed variants must contain no predictor NaNs.
    assert not processed_df[NODE_MODEL_FEATURES].isna().any().any()

    raw_X = numeric_predictors(raw_df).reset_index(drop=True)

    # Observed values must not be changed by imputation.
    for col in NODE_MODEL_FEATURES:
        observed = raw_X[col].notna()

        np.testing.assert_allclose(
            processed_df.loc[observed, col].astype(float),
            raw_X.loc[observed, col].astype(float),
            rtol=0,
            atol=1e-12
        )

        expected_indicator = raw_X[col].isna().astype('int8').reset_index(drop=True)

        actual_indicator = processed_df[f'{col}__missing'].astype('int8').reset_index(drop=True)

        assert expected_indicator.equals(actual_indicator)

    print(f'{variant_name}: audit passed ({len(processed_df):,} rows)')

In [ ]:
# ============================================================
# BUILD FINAL TABULAR DATA VARIANTS
# ============================================================

os.makedirs('./artifacts/final/full', exist_ok=True)
os.makedirs('./artifacts/preprocessing', exist_ok=True)

split_map = token_split[['token_mint', 'split']].copy()

final_tabular = {}
preprocessing_manifest = {}

for q in EXPERIMENTS:

    print(f'\n===== {q} =====')

    final_dir = f'./artifacts/final/full/{q}'
    prep_dir = f'./artifacts/preprocessing/{q}'

    os.makedirs(final_dir, exist_ok=True)
    os.makedirs(prep_dir, exist_ok=True)

    # Base candidate table.
    base = nodes_q[q][
        [
            'lp_address',
            'token_mint',
            'creation_time',
            'rug_pull_label'
        ]
        + NODE_MODEL_FEATURES
    ].copy()

    base['experiment_id'] = q

    base = base.merge(
        split_map,
        on='token_mint',
        how='left',
        validate='one_to_one'
    )

    assert base['split'].notna().all()

    # Embargo rows remain documented in the split manifest,
    # but are excluded from final model-development datasets.
    base = (
        base[
            base['split'].isin(
                ['train', 'validation', 'test']
            )
        ]
        .sort_values(
            ['creation_time', 'token_mint'],
            kind='mergesort'
        )
        .reset_index(drop=True)
    )

    assert not base['token_mint'].duplicated().any()

    X = numeric_predictors(base)

    train_mask = base['split'] == 'train'
    validation_mask = base['split'] == 'validation'
    test_mask = base['split'] == 'test'

    X_train = X.loc[train_mask]
    X_validation = X.loc[validation_mask]
    X_test = X.loc[test_mask]

    # Every candidate must have at least one observed training value.
    all_missing_train = [
        col
        for col in NODE_MODEL_FEATURES
        if X_train[col].notna().sum() == 0
    ]

    assert not all_missing_train, (f'{q}: entirely missing training predictors: {all_missing_train}')


    # ========================================================
    # Variant 1: RAW NaN
    # ========================================================

    raw_nan = base[
        [
            'lp_address',
            'token_mint',
            'experiment_id',
            'creation_time',
            'rug_pull_label',
            'split'
        ]
        + NODE_MODEL_FEATURES
    ].copy()

    raw_nan.to_parquet(f'{final_dir}/model_nodes_raw_nan.parquet', index=False)


    # ========================================================
    # Variant 2: TRAIN-MEDIAN + missing indicators
    # ========================================================

    median_imputer = SimpleImputer(strategy='median')

    median_imputer.fit(X_train)

    median_all = np.empty((len(X), len(NODE_MODEL_FEATURES)), dtype=float)

    median_all[train_mask.to_numpy()] = median_imputer.transform(X_train)
    median_all[validation_mask.to_numpy()] = median_imputer.transform(X_validation)
    median_all[test_mask.to_numpy()] = median_imputer.transform(X_test)

    median_indicator = rebuild_imputed_dataset(base, median_all)

    audit_imputed_variant(raw_nan, median_indicator, f'{q}/median_indicator')

    median_indicator.to_parquet(f'{final_dir}/model_nodes_median_indicator.parquet', index=False)

    joblib.dump(median_imputer, f'{prep_dir}/median_imputer.joblib')


    # Keep in memory for the rest of this notebook.
    final_tabular[q] = {
        'raw_nan': raw_nan,
        'median_indicator': median_indicator,
    }


    # Metadata for reproducibility.
    preprocessing_manifest[q] = {
        'features': NODE_MODEL_FEATURES,
        'indicator_columns': MISSING_INDICATOR_COLUMNS,

        'fit_partition': 'train_only',
        'label_used_for_imputation': False,

        'row_counts': {
            'train': int(train_mask.sum()),
            'validation': int(validation_mask.sum()),
            'test': int(test_mask.sum()),
        },

        'train_missing_percent': {
            col: float(X_train[col].isna().mean() * 100.0)
            for col in NODE_MODEL_FEATURES
        },

        'training_medians': {
            col: float(value)
            for col, value in zip(
                NODE_MODEL_FEATURES,
                median_imputer.statistics_
            )
        },
    }


with open('./artifacts/preprocessing/preprocessing_manifest.json', 'w', encoding='utf-8') as f:
    json.dump(preprocessing_manifest, f, indent=2)

print('\nBoth tabular variants saved.')

In [ ]:
# ============================================================
# FINAL GRAPH ARTIFACTS WITH THE SAME CENTRAL SPLIT
# ============================================================

for q in EXPERIMENTS:

    final_dir = f'./artifacts/final/full/{q}'

    # --------------------------------------------------------
    # Graph-summary modelling table
    # --------------------------------------------------------

    graph_model = gfeat[q][
        [
            'lp_address',
            'token_mint',
            'creation_time',
            'rug_pull_label'
        ]
        + GRAPH_MODEL_FEATURES
    ].copy()

    graph_model['experiment_id'] = q

    graph_model = graph_model.merge(
        split_map,
        on='token_mint',
        how='left',
        validate='one_to_one'
    )

    graph_model = graph_model[
        graph_model['split'].isin(
            ['train', 'validation', 'test']
        )
    ].reset_index(drop=True)

    graph_model.to_parquet(f'{final_dir}/model_graph_features_with_split.parquet', index=False)

    graph_token_set = set(graph_model['token_mint'])

    for variant_name, variant_df in final_tabular[q].items():

        matched_tabular = (
            variant_df[variant_df['token_mint'].isin(graph_token_set)]
            .copy()
            .sort_values(
                ['creation_time', 'token_mint'],
                kind='mergesort'
            )
            .reset_index(drop=True)
        )

        assert set(matched_tabular['token_mint']) == graph_token_set

        assert len(matched_tabular) == len(graph_model)

        matched_tabular.to_parquet(f'{final_dir}/model_nodes_{variant_name}_graph_matched.parquet', index=False)


    # --------------------------------------------------------
    # Recovered transfer rows
    # --------------------------------------------------------

    edge_final = edges_q[q].merge(
        split_map,
        on='token_mint',
        how='left',
        validate='many_to_one'
    )

    edge_final = edge_final[
        edge_final['split'].isin(
            ['train', 'validation', 'test']
        )
    ].reset_index(drop=True)

    edge_final.to_parquet(f'{final_dir}/edges_with_split.parquet', index=False)

    # Every graph row and edge row must use the same token assignment.
    assert graph_model.groupby('token_mint')['split'].nunique().max()<= 1

    if len(edge_final):
        assert edge_final.groupby('token_mint')['split'].nunique().max()<= 1

    print(q, 'graph samples =', len(graph_model), '| retained edges =', len(edge_final))

# Training-Only Model-Based Feature Importance

The chronological split and missing-data representations are now fixed.

Model-based feature importance is computed exactly once for each observation-window
population using only its training partition. The `raw_nan` representation is used so
that the importance ranking is not conditioned on the candidate imputation method.

The estimator is a histogram-based gradient-boosting classifier with native missing-value
support. Repeated permutation importance is measured using ROC-AUC. Validation and test
rows do not enter model fitting or permutation scoring.

These training-only rankings are exported as final experimental artifacts and are not
recomputed downstream.

## Feature-Importance Implementation Safeguards

The helper functions below implement the training-only feature-importance procedure.
They are defined at this point in the notebook, after the central chronological split
and missing-data artifacts have been constructed, and are called only on the training
partition.

The implementation fails explicitly if non-training rows are supplied. Numerical NaN
values are preserved because the histogram-based gradient-boosting estimator handles
missing values natively.

In [ ]:
# ============================================================
# TRAINING-ONLY FEATURE-IMPORTANCE HELPERS
# ============================================================
#
# These functions are defined here for reproducibility, but are
# called only AFTER the central chronological split is fixed.
#
# Feature importance must never be computed on the complete
# unsplit dataset.
# ============================================================


def _validate_training_only(df, *, data_role, split_col=None):
    """
    Fail fast unless the supplied data are explicitly training data.
    """

    if str(data_role).strip().lower() != 'train':
        raise ValueError("Feature importance is training-only. Pass data_role='train' and provide only the TRAIN partition.")

    effective_split_col = split_col

    if effective_split_col is None and 'split' in df.columns:
        effective_split_col = 'split'

    if effective_split_col is not None:

        if effective_split_col not in df.columns:
            raise KeyError(f"split_col {effective_split_col!r} is not present in the DataFrame.")

        observed = (
            df[effective_split_col]
            .dropna()
            .astype(str)
            .str.strip()
            .str.lower()
            .unique()
            .tolist()
        )

        non_train = sorted(
            value
            for value in observed
            if value != 'train'
        )

        if non_train:
            raise ValueError(f"Feature importance received non-training rows in {effective_split_col!r}: {non_train}")


def _validate_binary_label(y, label_col):
    """
    Validate that the training outcome contains both binary classes.
    """

    if y.isna().any():
        raise ValueError(f"{label_col!r} contains missing values.")

    unique = set(pd.unique(y))

    if not unique.issubset({0, 1}) or len(unique) != 2:
        raise ValueError(f"{label_col!r} must contain both binary classes 0 and 1; found {sorted(unique)}.")

    return y.astype(int)


def _prepare_numeric_features(df, feature_cols):
    """
    Select candidate predictors, require valid numeric values,
    preserve genuine NaN values, and reject observed infinities.

    NaN is retained because the estimator below handles missing
    numerical values natively.
    """

    missing_cols = [
        col
        for col in feature_cols
        if col not in df.columns
    ]

    if missing_cols:
        raise KeyError(f"Missing candidate feature columns: {missing_cols}")

    X = df.loc[:, list(feature_cols)].copy()

    for col in feature_cols:
        X[col] = pd.to_numeric(X[col], errors='raise')

    infinite_cols = [
        col
        for col in feature_cols
        if np.isinf(
            X[col]
            .dropna()
            .to_numpy(dtype=float)
        ).any()
    ]

    if infinite_cols:
        raise ValueError(f'Infinite predictor values found in: {infinite_cols}')

    all_missing = [
        col
        for col in feature_cols
        if X[col].notna().sum() == 0
    ]

    if all_missing:
        raise ValueError(f"These candidate features are entirely missing in the supplied training partition: {all_missing}")

    return X


def compute_training_feature_importance(
        train_df,
        feature_cols,
        *,
        data_role,
        label_col='rug_pull_label',
        split_col=None,
        n_repeats=30,
        random_state=42,
        learning_rate=0.05,
        max_iter=250,
        max_leaf_nodes=15,
        min_samples_leaf=20,
        l2_regularization=1.0
):
    """
    Fit the feature-importance model using TRAINING DATA ONLY.

    HistGradientBoostingClassifier is used because it handles
    numerical NaN values natively.

    Importance is estimated with repeated permutation importance
    using ROC-AUC.
    """

    _validate_training_only(
        train_df,
        data_role=data_role,
        split_col=split_col
    )

    if label_col not in train_df.columns:
        raise KeyError(f"Missing label column: {label_col!r}")

    if len(feature_cols) == 0:
        raise ValueError("feature_cols must contain at least one feature.")

    if n_repeats < 2:
        raise ValueError("n_repeats must be at least 2.")

    work = train_df.loc[:, list(feature_cols) + [label_col]].copy()

    y = _validate_binary_label(work[label_col], label_col)

    X = _prepare_numeric_features(work, feature_cols)

    model = HistGradientBoostingClassifier(
        learning_rate=learning_rate,
        max_iter=max_iter,
        max_leaf_nodes=max_leaf_nodes,
        min_samples_leaf=min_samples_leaf,
        l2_regularization=l2_regularization,
        early_stopping=False,
        random_state=random_state
    )

    model.fit(X,y)

    perm = permutation_importance(
        model,
        X,
        y,
        scoring='roc_auc',
        n_repeats=n_repeats,
        random_state=random_state,
        n_jobs=-1
    )

    importance_df = pd.DataFrame(
        {
            'feature': list(feature_cols),

            'importance_mean': perm.importances_mean,

            'importance_std': perm.importances_std,

            'non_missing_n': [
                int(X[col].notna().sum())
                for col in feature_cols
            ],

            'missing_pct': [
                float(X[col].isna().mean() * 100.0)
                for col in feature_cols
            ]
        }
    )

    importance_df = (
        importance_df
        .sort_values(
            [
                'importance_mean',
                'feature'
            ],
            ascending=[
                False,
                True
            ],
            kind='mergesort'
        )
        .reset_index(drop=True)
    )

    importance_df['rank'] = np.arange(1, len(importance_df) + 1)

    importance_df = importance_df[
        [
            'rank',
            'feature',
            'importance_mean',
            'importance_std',
            'non_missing_n',
            'missing_pct'
        ]
    ]

    return importance_df, model


def ranked_feature_names(importance_df):
    """
    Return feature names ordered from highest to lowest
    training-only permutation importance.
    """

    required = {
        'feature',
        'importance_mean'
    }

    missing = required.difference(importance_df.columns)

    if missing:
        raise KeyError(f"importance_df is missing columns: {sorted(missing)}")

    ranked = (
        importance_df
        .sort_values(
            [
                'importance_mean',
                'feature'
            ],
            ascending=[
                False,
                True
            ],
            kind='mergesort'
        )
    )

    return ranked['feature'].tolist()


def build_top_k_feature_sets(importance_df, *, min_k=1, include_full_set=True):
    """
    Construct candidate top-k feature subsets from the fixed
    training-only ranking.

    This function does NOT select the final feature subset.
    Candidate subsets are later compared using validation data.
    """

    ranked = ranked_feature_names(importance_df)

    p = len(ranked)

    if not 1 <= min_k <= p:
        raise ValueError(f"min_k must be between 1 and {p}.")

    ks = list(range(min_k, p + 1))

    if not include_full_set and p in ks:
        ks.remove(p)

    return {f'top_{k}': ranked[:k] for k in ks}


print('Training-only feature-importance helpers defined.')

## Compute and Export Training-Only Importance

In [ ]:
# ============================================================
# TRAINING-ONLY FEATURE IMPORTANCE
# ============================================================

os.makedirs('./artifacts/feature_importance', exist_ok=True)

feature_importance_by_q = {}
candidate_feature_sets = {}

for q in EXPERIMENTS:

    train_raw = (
        final_tabular[q]['raw_nan']
        .loc[
            lambda df: df['split'] == 'train'
        ]
        .copy()
    )

    importance_df, _ = (
        compute_training_feature_importance(
            train_raw,
            NODE_MODEL_FEATURES,
            data_role='train',
            split_col='split',
            label_col='rug_pull_label',
            n_repeats=30,
            random_state=42
        )
    )

    importance_df.insert(0, 'experiment_id',q)

    feature_importance_by_q[q] = importance_df.copy()

    importance_df.to_csv(f'./artifacts/feature_importance/{q}_training_only.csv', index=False)

    original_sets = build_top_k_feature_sets(importance_df, min_k=1, include_full_set=True)

    candidate_feature_sets[q] = {
        'raw_nan': original_sets,

        'median_indicator': {
            name: (
                features
                + [
                    f'{feature}__missing'
                    for feature in features
                ]
            )
            for name, features
            in original_sets.items()
        }
    }

    print(f'\n{q} training-only feature importance')
    display(importance_df)


feature_importance_combined = pd.concat(
    [
        feature_importance_by_q[q]
        for q in EXPERIMENTS
    ],
    ignore_index=True
)

feature_importance_combined.to_csv('./artifacts/feature_importance/training_only_combined.csv', index=False)

with open('./artifacts/feature_importance/candidate_feature_sets.json', 'w', encoding='utf-8') as f:
    json.dump(candidate_feature_sets, f, indent=2)

In [ ]:
# ============================================================
# TRAINING-ONLY FEATURE-IMPORTANCE FIGURE
# ============================================================

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))

for ax, q in zip(axes, EXPERIMENTS):

    plot_df = feature_importance_by_q[q].sort_values('importance_mean', ascending=True)

    ax.barh(
        plot_df['feature'],
        plot_df['importance_mean'],
        xerr=plot_df['importance_std'],
        capsize=3
    )

    ax.axvline(0, linewidth=0.8)
    ax.set_title(f'{q} — training only')
    ax.set_xlabel('Permutation importance\n(mean ROC-AUC decrease)')

fig.suptitle('Training-only model-based feature importance')

fig.tight_layout()

fig.savefig('./artifacts/figures/training_only_feature_importance.png', dpi=300, bbox_inches='tight')
fig.savefig('./artifacts/figures/training_only_feature_importance.pdf', bbox_inches='tight')

plt.show()

In [ ]:
# ============================================================
# FINAL MODEL-READY ARTIFACT AUDIT
# ============================================================

audit_rows = []

for q in EXPERIMENTS:

    for variant_name, df in final_tabular[q].items():

        assert set(df['split'].unique()) == {'train', 'validation', 'test'}

        for split_name in [
            'train',
            'validation',
            'test'
        ]:

            part = (df[df['split'] == split_name].copy())

            missing_cells = int(part[NODE_MODEL_FEATURES].isna().sum().sum())

            total_predictor_cells = len(part) * len(NODE_MODEL_FEATURES)

            missing_cell_pct = 100.0 * missing_cells / total_predictor_cells if total_predictor_cells else np.nan

            audit_rows.append(
                {
                    'experiment_id': q,
                    'variant': variant_name,
                    'split': split_name,
                    'rows': len(part),

                    'rug_rate_percent': part['rug_pull_label'].mean() * 100.0,

                    'predictor_missing_cells': missing_cells,

                    'predictor_missing_cell_pct': missing_cell_pct
                }
            )


final_artifact_audit = pd.DataFrame(audit_rows)


# ------------------------------------------------------------
# Strong audits for imputed variants
# ------------------------------------------------------------

for q in EXPERIMENTS:

    assert set(final_tabular[q].keys()) == {
        'raw_nan',
        'median_indicator'
    }

    raw_ids = (
        final_tabular[q]['raw_nan'][
            [
                'lp_address',
                'token_mint',
                'experiment_id',
                'creation_time',
                'rug_pull_label',
                'split'
            ]
        ]
        .reset_index(drop=True)
    )

    for variant_name in ['median_indicator']:

        df = final_tabular[q][variant_name]

        # No missing values may remain in the six original predictors.
        assert not df[NODE_MODEL_FEATURES].isna().any().any()

        # No infinite values.
        assert np.isfinite(df[NODE_MODEL_FEATURES].to_numpy(dtype=float)).all()

        # Row identity and outcomes must be unchanged.
        processed_ids = (
            df[
                [
                    'lp_address',
                    'token_mint',
                    'experiment_id',
                    'creation_time',
                    'rug_pull_label',
                    'split'
                ]
            ]
            .reset_index(drop=True)
        )

        assert raw_ids.equals(processed_ids)

        # Indicators must exist and be binary.
        for indicator in MISSING_INDICATOR_COLUMNS:

            assert indicator in df.columns

            assert set(df[indicator].dropna().unique()).issubset({0, 1})


final_artifact_audit.to_csv('./artifacts/final/final_artifact_audit.csv', index=False)

display(final_artifact_audit)

In [ ]:
# ============================================================
# SOFTWARE VERSION MANIFEST
# ============================================================

software_versions = {
    'python': sys.version.split()[0],
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'scikit_learn': sklearn.__version__,
    'joblib': joblib.__version__,
    'networkx': nx.__version__,
}

with open('./artifacts/software_versions.json', 'w', encoding='utf-8') as f:
    json.dump(software_versions, f, indent=2)

print(json.dumps(software_versions, indent=2))

# Summary

The notebook now completes the full data-preparation workflow required before model
training.

Completed steps:

* cleaned and audited token-level, recovered-edge, and label tables;
* q1/q2/q3 observation-window populations;
* complete-data descriptive EDA and temporal diagnostics;
* token-level missingness, univariate association, correlation, authority, and creator
  diagnostics;
* recovered-graph availability, topology, structural, and univariate diagnostics;
* label-timing, predictor-provenance, and direct-leakage audits;
* one central label-independent chronological split at the token-mint level;
* temporal embargo checks based on `label_available_time`;
* one common split assignment reused across q1/q2/q3 and across tabular and graph branches;
* two final token-level missing-data representations for each observation window:
  `raw_nan` and `median_indicator`;
* training-fitted median imputers saved for reproducibility;
* training-only model-based feature importance for all six token-level candidate features;
* candidate top-k feature-set definitions generated from the fixed training-only rankings;
* final split-tagged graph-summary and recovered-edge artifacts;
* final artifact-level missingness and partition audits.

The complete-data EDA does not use the model-development split for descriptive feature
analysis. Conversely, learned imputation and model-based feature importance are performed
only after the chronological split is fixed.

Validation and test rows do not influence fitted imputation parameters or the
feature-importance ranking. Embargoed tokens remain documented in the split manifest but
are excluded from final model-development datasets.

The primary final tabular artifacts are stored under:

`artifacts/final/full/q1--q3/`

with one file for each missing-data representation. The same directories contain the
split-tagged graph-summary and recovered-edge artifacts.

The central source of truth for partition assignment is:

`artifacts/splits/chronological_split_manifest.parquet`

Training-only feature-importance results are stored under:

`artifacts/feature_importance/`

Remaining modelling constraint:

* amount-based graph features remain excluded from the graph-model candidate set until
  transfer-amount scaling/provenance is represented consistently at the edge-row level.